# V4.5.0 Seven Class: Fresh Training with Outlier Exposure

This experimental notebook builds a new mango classifier from the original images and ImageNet features. It never loads a previous MangoLeaf model, split, mask, threshold file, or run folder. The 62 embedded reviewed image fingerprints describe original photographs; they are not inherited training membership.

**Outlier Exposure (OE)** means training on a separate collection of unsupported photos while encouraging similar probabilities across all seven mango classes. Mango photos still learn their correct class. Unsupported photos do not become an eighth class. The OE contribution is an experimental setting, not a guarantee that unfamiliar objects will be rejected.

The six augmentation categories are flip, rotation, zoom, shear, global brightness, and random contrast 0.1. Background replacement and localized lighting are absent. Training uses five epochs at learning rate 0.001, then ten at 0.00001. Temperature is fitted using mango validation photos only. A separate threshold notebook chooses rejection settings and evaluates held-out photos after freezing those settings.

Run ordinary Colab Python cells in order on a GPU runtime signed into the account whose My Drive will store the new results. Shared original photo folders are supported through explicitly enabled My Drive shortcuts. There is no custom cell magic or hidden scientific kernel. Inspect the review sheets before changing any review decision to True. Review gates deliberately stop until their notes and decisions are complete; rerun that same cell after editing them. Do not rerun the run-creation cell during a review pause.

This file is an unexecuted template. It contains no measured v4.5.0 accuracy, confusion counts, or claimed browser parity. The app's adopted v4.3.0 assets remain separate.


## Step 1: Mount Drive and install the explicit training runtime

**What:** Mount your own Google Drive and install only the packages chosen for this experiment.

**Why:** Normal Colab cells must use one known scientific runtime. Historical whole-environment snapshots are evidence, never installation instructions.

**How:** Before running code, open Runtime > Change runtime type. Keep Python 3, select a GPU such as T4, and choose Runtime Version 2026.04. Google's runtime list identifies that image as Python 3.12.13 with TensorFlow 2.19.0. The Python 3 label and GPU choice alone do not select a compatible Python minor version.

If 2026.04 is unavailable, select a listed runtime with Python 3.11 or 3.12 and check the exact version printed by this cell. See the [official Colab runtime list](https://research.google.com/colaboratory/runtime-version-faq.html). Do not remove the Python check or upgrade TensorFlow to bypass it.

This cell updates pip first, then installs the listed scientific and security versions once and records its process ID. Explicit helper pins prevent already-satisfying older Colab packages from being retained. After installation, use Runtime > Restart session, then rerun Step 1 before Step 2. The converter will use a separate CPU environment later.

**Expected result:** Drive is mounted and the new runtime receipt says whether a restart is needed. No mango model is trained.


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

# Check Python before mounting Drive or installing scientific packages.
print('Notebook Python:', sys.version.split()[0])
if sys.version_info[:2] not in {(3, 11), (3, 12)}:
    raise RuntimeError(
        'Unsupported Python ' + sys.version.split()[0] + '. '
        'Open Runtime > Change runtime type, keep a GPU such as T4, and select '
        'Runtime Version 2026.04 (Python 3.12.13), then rerun Step 1. '
        'If unavailable, use a listed Python 3.11/3.12 runtime: '
        'https://research.google.com/colaboratory/runtime-version-faq.html. '
        'These TensorFlow 2.19.0 pins must not run on an unsupported Python version.'
    )

from google.colab import drive

drive.mount('/content/drive')

TRAINING_PACKAGES = [
    'tensorflow==2.19.0',
    'keras==3.15.1',
    'numpy==2.1.3',
    'Pillow==12.3.0',
    'ImageHash==4.3.2',
    'scipy==1.16.3',
    'matplotlib==3.11.2',
    'scikit-learn==1.9.1',
    'pip-audit==2.10.1',
    'packaging==25.0',
    'uv==0.12.23',
    # Explicit security pins prevent pip from retaining vulnerable provider helpers.
    'pip==26.2.1',
    'setuptools==83.0.0',
    'idna==3.15',
    'msgpack==1.2.1',
    'requests==2.33.0',
    'urllib3==2.8.0',
    'werkzeug==3.1.9',
]
setup_directory = Path('/content/mangoleaf-v450-setup')
setup_directory.mkdir(exist_ok=True)
setup_receipt = setup_directory / 'installation.json'
installed = json.loads(setup_receipt.read_text()) if setup_receipt.is_file() else None

if installed is None:
    # Upgrade the installer first; this subprocess does not import scientific modules.
    pip_pin = next(pin for pin in TRAINING_PACKAGES if pin.startswith('pip=='))
    bootstrap = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
         '--only-binary=:all:', '--no-deps', pip_pin],
        capture_output=True, text=True, check=False,
    )
    (setup_directory / 'pip-bootstrap.log').write_text(bootstrap.stdout + '\n' + bootstrap.stderr)
    if bootstrap.returncode != 0:
        raise RuntimeError('Installer update failed. Read: ' + str(setup_directory / 'pip-bootstrap.log'))
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
         '--only-binary=:all:', *TRAINING_PACKAGES],
        capture_output=True, text=True, check=False,
    )
    (setup_directory / 'installation.log').write_text(result.stdout + '\n' + result.stderr)
    if result.returncode != 0:
        raise RuntimeError('Package installation failed. Read: ' + str(setup_directory / 'installation.log'))
    setup_receipt.write_text(json.dumps({
        'packages': TRAINING_PACKAGES, 'installationPid': os.getpid(),
        'python': sys.version, 'restartRequired': True,
    }, indent=2) + '\n')
    print('Installation complete. Restart session, then rerun Step 1 and continue.')
else:
    if installed['packages'] != TRAINING_PACKAGES:
        raise RuntimeError('Setup pins changed. Use a fresh runtime rather than mixing installation attempts.')
    print('Packages already installed. Restart completed:', installed['installationPid'] != os.getpid())


## Step 2: Verify versions, dependencies, and real GPU execution

**What:** Check the packages actually imported and run a small operation on the GPU.

**Why:** A successful install is not proof of compatibility or GPU use. Dependency findings must stay visible.

**How:** Reject a session that has not restarted. Check exact pins, audit the installed dependency closure of the training packages, record provider package conflicts separately, and require a real GPU operation. Audit failures print affected packages, advisory IDs, available fixes, and collection gaps before stopping. A nonzero vulnerability result must be fixed rather than ignored.

**Expected result:** A runtime record and dependency audit are saved locally. No image predictions or training epochs occur.


In [ ]:
import csv
import hashlib
import importlib.metadata
import math
import random
import shutil
import uuid
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import PurePosixPath

# Stop before scientific imports can mix pre-install modules with new files.
installed = json.loads(setup_receipt.read_text())
if installed['installationPid'] == os.getpid():
    raise RuntimeError('Restart the Colab session after installation, rerun Step 1, then rerun this cell.')

import imagehash
import matplotlib.pyplot as plt
import numpy as np
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name
from PIL import Image, ImageDraw, ImageOps
from scipy.optimize import minimize_scalar
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf

for pin in TRAINING_PACKAGES:
    name, expected = pin.split('==')
    if importlib.metadata.version(name) != expected:
        raise RuntimeError('Runtime version differs from chosen pin: ' + pin)
if tf.__version__ != '2.19.0' or tf.keras.__version__ != '3.15.1':
    raise RuntimeError('Imported TensorFlow/Keras versions differ from installed pins; restart a clean session.')

# Audit only the transitive research runtime, rather than treating every
# unrelated preinstalled Colab provider package as an executed experiment dependency.
def installed_research_dependencies(pins):
    names = set()
    examined_contexts = set()
    pending_requirements = [Requirement(pin) for pin in pins]
    while pending_requirements:
        requirement = pending_requirements.pop()
        name = canonicalize_name(requirement.name)
        distribution = importlib.metadata.distribution(name)
        if not requirement.specifier.contains(distribution.version, prereleases=True):
            raise RuntimeError(
                'Research dependency conflict: ' + str(requirement)
                + '; installed ' + distribution.version
            )
        names.add(name)
        # Extras such as CacheControl[filecache] add dependencies that must be audited.
        # Revisit a package when a later requirement enables a different extra.
        for extra in {''} | requirement.extras:
            context = (name, extra)
            if context in examined_contexts:
                continue
            examined_contexts.add(context)
            for dependency in distribution.requires or []:
                child = Requirement(dependency)
                if child.marker is None or child.marker.evaluate({'extra': extra}):
                    pending_requirements.append(child)
    return names


research_names = installed_research_dependencies(TRAINING_PACKAGES)

runtime_lock = setup_directory / 'training-runtime-lock.txt'
runtime_lock.write_text(''.join(
    name + '==' + importlib.metadata.version(name) + '\n'
    for name in sorted(research_names)
), encoding='utf-8')
audit_path = setup_directory / 'training-audit.json'
audit = subprocess.run([
    sys.executable, '-m', 'pip_audit', '-r', str(runtime_lock),
    '--no-deps', '--disable-pip', '--format=json', '--output', str(audit_path),
    '--progress-spinner=off',
], capture_output=True, text=True, check=False)
(setup_directory / 'training-audit.log').write_text(audit.stdout + '\n' + audit.stderr)
audit_report = json.loads(audit_path.read_text()) if audit_path.is_file() else {}
dependencies = audit_report.get('dependencies', [])
findings = [row for row in dependencies if row.get('vulns')]
skipped = [row for row in dependencies if row.get('skip_reason')]
audited_names = {canonicalize_name(row['name']) for row in dependencies if row.get('name')}
if audit.returncode != 0 or findings or skipped or audited_names != research_names:
    print('Dependency audit exit:', audit.returncode)
    for row in findings:
        print('Unresolved package:', row['name'], row.get('version', 'unknown'))
        details = sorted({
            vulnerability.get('id', 'unknown') + ' (fix versions: '
            + ', '.join(vulnerability.get('fix_versions', [])) + ')'
            for vulnerability in row['vulns']
        })
        for detail in details:
            print('  ', detail)
    for row in skipped:
        print('Skipped package:', row.get('name', 'unknown'), row['skip_reason'])
    print('Missing packages:', sorted(research_names - audited_names))
    print('Unexpected packages:', sorted(audited_names - research_names))
    print('Full audit and log:', audit_path, setup_directory / 'training-audit.log')
    raise RuntimeError('Training dependency audit incomplete or unresolved. Read ' + str(audit_path))

provider_check = subprocess.run(
    [sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True, check=False,
)
(setup_directory / 'provider-pip-check.log').write_text(provider_check.stdout + '\n' + provider_check.stderr)
if provider_check.returncode:
    print('Colab provider compatibility findings remain visible:', setup_directory / 'provider-pip-check.log')

gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError('Select a Colab GPU runtime before training.')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
with tf.device('/GPU:0'):
    gpu_probe = tf.linalg.matmul(tf.ones((16, 16)), tf.ones((16, 16)))
if 'GPU' not in gpu_probe.device or not np.isfinite(gpu_probe.numpy()).all():
    raise RuntimeError('The GPU execution probe failed.')
RUNTIME = {
    'python': sys.version, 'tensorflow': tf.__version__,
    'packages': {name: importlib.metadata.version(name) for name in sorted(research_names)},
    'trainingDependencyAuditPassed': True, 'providerPipCheckReturnCode': provider_check.returncode,
    'gpuCount': len(gpus), 'gpuProbeDevice': gpu_probe.device,
    'execution': 'ordinary Colab Python cells; no custom cell magic',
    'browserParityPassed': False,
}
print('TensorFlow:', tf.__version__, 'Keras:', tf.keras.__version__, 'GPU count:', len(gpus))


## Step 3: Choose the original inputs and create one fresh output run

**What:** Select the raw mango, OE, and held-out OOD folders through My Drive, and choose this account's actual output folder.

**Why:** Fresh outputs keep previous experiments intact. Shared input shortcuts can resolve to the original photos, while results must stay in the selected account's actual My Drive folder.

**How:** Use actual input folders under My Drive, or add shortcuts to shared photo folders and set ALLOW_SHARED_INPUT_SHORTCUTS = True after checking their targets. Keep OUTPUT_ROOT in the actual mangoleaf_experiments folder owned by the mounted account. If two folders have the same name, rename the shared output shortcut so the selected output path is unambiguous. Unknown OE count is discovered later. The first run requires all 500 held-out OOD photos: upload 300 fresh photos alongside the existing 200 before running inventory. Only a new run directory is created; input photos are never moved or modified.

**Expected result:** The cell prints each resolved folder and one new v4.5.0 run path. Save that exact path for threshold fitting.


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
RAW_MANGO_INPUT = DRIVE_ROOT / 'final_mangoleafai_dataset_public_and_bpi'
OE_INPUT = DRIVE_ROOT / 'mangoleafai_oe_images'
OOD_INPUT = DRIVE_ROOT / 'mango_ood_test_v4_4_0'
OUTPUT_ROOT = DRIVE_ROOT / 'mangoleaf_experiments/v4.5.0/training'
# Set True only when the selected shared photo folders have My Drive shortcuts.
ALLOW_SHARED_INPUT_SHORTCUTS = False

EXPERIMENT_VERSION = 'v4.5.0'
MODEL_VERSION = 'model-v4.5.0-candidate'
CLASS_NAMES = [
    'anthracnose', 'bacterial_black_spot', 'bacterial_canker',
    'healthy_fruit', 'healthy_leaf', 'mango_scab', 'sooty_mold',
]
SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EXPECTED_RAW_IMAGE_COUNT = 5098
EXPECTED_OOD_IMAGE_COUNT = 500  # Existing 200 plus 300 fresh photos, uploaded before training.
PHASH_DISTANCE = 5
STAGE1_EPOCHS = 5
STAGE2_EPOCHS = 10
STAGE1_LEARNING_RATE = 0.001
STAGE2_LEARNING_RATE = 0.00001
OE_LOSS_WEIGHT = 0.5  # Experimental coefficient; not a fitted optimum.
CONTRAST_FACTOR = 0.1

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
def drive_input_path(path, allow_shared=False, my_drive=Path('/content/drive/MyDrive')):
    """Accept selected My Drive inputs, with explicit shared-shortcut permission."""
    selected = Path(path)
    my_drive = Path(my_drive)
    if (not selected.is_absolute() or '..' in selected.parts
            or selected == my_drive or not selected.is_relative_to(my_drive)):
        raise ValueError('Select an input folder through a path below My Drive: ' + str(selected))

    resolved = selected.resolve()
    owned_root = my_drive.resolve()
    shortcut_root = (my_drive.parent / '.shortcut-targets-by-id').resolve()
    owned = resolved != owned_root and resolved.is_relative_to(owned_root)
    shared = (resolved.is_relative_to(shortcut_root)
              and len(resolved.relative_to(shortcut_root).parts) >= 2)
    print('Input:', selected, '\nResolved:', resolved)
    if not owned and not (allow_shared and shared):
        raise ValueError(
            'Input resolves outside My Drive. For a reviewed shared input, add its shortcut '
            'to My Drive and set ALLOW_SHARED_INPUT_SHORTCUTS = True. '
            'Other external paths remain unsupported: ' + str(resolved)
        )
    if not resolved.is_dir():
        raise FileNotFoundError('Input folder missing or inaccessible in the mounted account: ' + str(selected))
    return selected


def owned_my_drive_path(path, my_drive=Path('/content/drive/MyDrive')):
    """Keep outputs and completed training runs in the mounted account's My Drive."""
    selected = Path(path)
    resolved = selected.resolve()
    owned_root = Path(my_drive).resolve()
    print('Selected location:', selected)
    print('Resolved location:', resolved)
    print('Exists:', resolved.exists())
    if resolved == owned_root or not resolved.is_relative_to(owned_root):
        raise ValueError(
            'Output and training-run paths must resolve inside this mounted account\'s '
            'My Drive. Select its actual mangoleaf_experiments folder, not a shortcut '
            'to another account\'s output folder: ' + str(resolved)
        )
    return resolved


for input_path in (RAW_MANGO_INPUT, OE_INPUT, OOD_INPUT):
    drive_input_path(input_path, allow_shared=ALLOW_SHARED_INPUT_SHORTCUTS)
OUTPUT_ROOT = owned_my_drive_path(OUTPUT_ROOT)
inputs_resolved = [p.resolve() for p in (RAW_MANGO_INPUT, OE_INPUT, OOD_INPUT)]
for index, first in enumerate(inputs_resolved):
    for second in inputs_resolved[index + 1:] + [OUTPUT_ROOT.resolve()]:
        if first == second or first.is_relative_to(second) or second.is_relative_to(first):
            raise ValueError('Each input and output location must be separate.')

# Re-running this cell creates a different run. Review pauses resume at their
# own cell rather than recreating the run and losing the active evidence path.
run_id = 'v4-5-0-training-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
RUN_DIR = OUTPUT_ROOT / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
DATASET_DIR = RUN_DIR / 'dataset'
FIGURE_DIR = RUN_DIR / 'figures'
FIGURE_DIR.mkdir()
shutil.copytree(setup_directory, RUN_DIR / 'runtime_preflight')
(RUN_DIR / 'runtime.json').write_text(json.dumps(RUNTIME, indent=2) + '\n')
print('Preserve this exact run directory:', RUN_DIR)


## Step 4: Inventory original mango, OE, and held-out OOD photos

**What:** Count and fingerprint every readable image without making predictions.

**Why:** Unknown OE counts and mislabeled or unreadable files must be resolved before training. OOD identities are recorded now to detect leakage into OE training.

**How:** First count supported image files, checking the required 500 OOD photos before any mango/OE decoding. If that collection is incomplete, stop with its exact path and count. Then scan each input, record file/decoded hashes, perceptual hashes, dimensions, and relative names. Print progress every 100 photos and every ten OE preview pages. A normal rerun starts the inventories again; it has no resume checkpoint. Mango class folders must exactly match the seven listed labels. Create contact sheets for OE inspection.

**Expected result:** Three fresh inventories and actual counts. No OE count is invented and no additional OOD photos are manufactured.


In [ ]:
import time

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'}
PREPROCESSING = 'RGB decode; bilinear 224x224 without antialias; float32 0-255; MobileNetV3 internal rescaling'


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def object_sha256(value):
    serialized = json.dumps(value, sort_keys=True, separators=(',', ':'), allow_nan=False)
    return hashlib.sha256(serialized.encode('utf-8')).hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def save_json_once(path, value):
    path = Path(path)
    if path.exists():
        if read_json(path) != value:
            raise RuntimeError('Refusing to overwrite changed evidence: ' + str(path))
        return
    with path.open('x', encoding='utf-8', newline='\n') as handle:
        json.dump(value, handle, indent=2, ensure_ascii=False, allow_nan=False)
        handle.write('\n')


def safe_child(root, relative):
    if not isinstance(relative, str) or not relative or '\\' in relative or ':' in relative or '\x00' in relative:
        raise ValueError('Unsafe relative path: ' + str(relative))
    child = PurePosixPath(relative)
    if child.is_absolute() or '..' in child.parts or child == PurePosixPath('.'):
        raise ValueError('Unsafe relative path: ' + relative)
    destination = Path(root).joinpath(*child.parts).resolve()
    destination.relative_to(Path(root).resolve())
    return destination


def raw_rgb(path):
    with Image.open(path) as opened:
        return ImageOps.exif_transpose(opened).convert('RGB')


def load_pixels(path):
    # TensorFlow matches the app's bilinear float32 input. Pillow handles WebP
    # content, including WebP files whose suffix claims JPEG.
    with Path(path).open('rb') as handle:
        signature = handle.read(12)
    if signature[:4] == b'RIFF' and signature[8:12] == b'WEBP':
        image = tf.convert_to_tensor(np.asarray(raw_rgb(path)), dtype=tf.uint8)
    else:
        image = tf.io.decode_image(tf.io.read_file(str(path)), channels=3, expand_animations=False)
    return tf.image.resize(tf.cast(image, tf.float32), IMAGE_SIZE, method='bilinear', antialias=False)


def portable_rows(rows):
    return [{key: value for key, value in row.items() if key != 'path'} for row in rows]


def verify_image_rows(rows):
    for row in rows:
        if file_sha256(row['path']) != row['sha256']:
            raise RuntimeError('Input image changed after inventory: ' + row['path'])


def image_paths(folder):
    return sorted(
        path for path in Path(folder).rglob('*')
        if path.suffix.lower() in IMAGE_EXTENSIONS and path.is_file()
    )


def checked_inventory_paths(mango_folder, oe_folder, ood_folder):
    # Check the planned OOD count before decoding thousands of mango/OE photos.
    print('Counting OOD image files:', ood_folder, flush=True)
    ood_paths = image_paths(ood_folder)
    if len(ood_paths) != EXPECTED_OOD_IMAGE_COUNT:
        raise ValueError(
            'Expected ' + str(EXPECTED_OOD_IMAGE_COUNT) + ' held-out OOD image files; counted '
            + str(len(ood_paths)) + ' in ' + str(ood_folder)
            + '. Complete the planned OOD collection, then rerun Step 4. No image decoding has started.'
        )
    print('Counting mango image files:', mango_folder, flush=True)
    mango_paths = image_paths(mango_folder)
    if len(mango_paths) != EXPECTED_RAW_IMAGE_COUNT:
        raise ValueError('Expected ' + str(EXPECTED_RAW_IMAGE_COUNT) + ' original mango files; counted ' + str(len(mango_paths)))
    print('Counting OE image files:', oe_folder, flush=True)
    oe_paths = image_paths(oe_folder)
    if not oe_paths:
        raise ValueError('The OE folder contains no supported image files.')
    print('Count preflight passed. Mango:', len(mango_paths), 'OE:', len(oe_paths), 'OOD:', len(ood_paths), flush=True)
    return mango_paths, oe_paths, ood_paths


def image_inventory(folder, pool, labelled=False, paths=None):
    rows = []
    failures = []
    paths = image_paths(folder) if paths is None else paths
    started = time.monotonic()
    print(pool, 'inventory started:', len(paths), 'images', flush=True)
    for index, path in enumerate(paths, start=1):
        relative = path.relative_to(folder).as_posix()
        try:
            pixels = raw_rgb(path)
            loaded = load_pixels(path).numpy()
            if loaded.shape != (224, 224, 3) or not np.isfinite(loaded).all():
                raise ValueError('Invalid decoded model input')
            sha = file_sha256(path)
            decoded = hashlib.sha256(str(pixels.size).encode('utf-8') + pixels.tobytes()).hexdigest()
            row = {
                'id': sha[:20], 'pool': pool, 'path': str(path),
                'sourceRelativePath': relative, 'sha256': sha,
                'decodedRgbSha256': decoded, 'bytes': path.stat().st_size,
                'phash': str(imagehash.phash(pixels, hash_size=8)),
                'dimensions': list(pixels.size),
            }
            if labelled:
                row['label'] = PurePosixPath(relative).parts[0]
                if row['label'] not in CLASS_NAMES:
                    raise ValueError('Unexpected class folder: ' + row['label'])
            rows.append(row)
        except Exception as error:
            failures.append({'relativePath': relative, 'error': str(error)})
        if index % 100 == 0 or index == len(paths):
            print(pool, 'images processed:', index, '/', len(paths),
                  'elapsed seconds:', round(time.monotonic() - started, 1), flush=True)
    if failures:
        save_json_once(RUN_DIR / (pool + '_inventory_failures.json'), failures)
        raise RuntimeError('Correct unreadable or mislabeled files. First failures: ' + json.dumps(failures[:10]))
    identities = defaultdict(set)
    for row in rows:
        identities[row['id']].add(row['sha256'])
    if any(len(values) != 1 for values in identities.values()):
        raise RuntimeError('Truncated image ID collision. Use full SHA-256 IDs for a new run.')
    return rows


def require_completed_inventories():
    if len(raw_rows) != EXPECTED_RAW_IMAGE_COUNT or not oe_raw_rows or len(held_out_ood_rows) != EXPECTED_OOD_IMAGE_COUNT:
        raise RuntimeError('Step 4 inventories are incomplete. Finish its required counts before Step 5; uploading files alone does not refresh notebook variables.')
    for filename, rows in (
        ('mango_inventory.json', raw_rows), ('oe_raw_inventory.json', oe_raw_rows),
        ('held_out_ood_raw_inventory.json', held_out_ood_rows),
    ):
        path = RUN_DIR / filename
        if not path.is_file() or read_json(path) != {'images': portable_rows(rows)}:
            raise RuntimeError('Step 4 inventory evidence is missing or differs from memory: ' + str(path))


def save_contact_sheets(rows, destination, title, columns=4):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    page_count = math.ceil(len(rows) / 20)
    print(destination.name, 'preview sheets started:', page_count, 'pages', flush=True)
    for page, start in enumerate(range(0, len(rows), 20), start=1):
        chosen = rows[start:start + 20]
        sheet = Image.new('RGB', (columns * 280, math.ceil(len(chosen) / columns) * 295 + 40), 'white')
        drawing = ImageDraw.Draw(sheet)
        drawing.text((8, 8), title, fill='black')
        for index, row in enumerate(chosen):
            picture = raw_rgb(row['path'])
            picture.thumbnail((268, 240))
            left = (index % columns) * 280 + 6
            top = (index // columns) * 295 + 40
            sheet.paste(picture, (left, top))
            drawing.text((left, top + 242), f"Photo {start + index + 1}: {row['id']}\n{row['sourceRelativePath'][:38]}", fill='black')
        sheet.save(destination / f'page_{page:03d}.png')
        if page % 10 == 0 or page == page_count:
            print(destination.name, 'preview pages saved:', page, '/', page_count, flush=True)


mango_paths, oe_paths, ood_paths = checked_inventory_paths(RAW_MANGO_INPUT, OE_INPUT, OOD_INPUT)
raw_rows = image_inventory(RAW_MANGO_INPUT, 'mango', labelled=True, paths=mango_paths)
oe_raw_rows = image_inventory(OE_INPUT, 'oe', paths=oe_paths)
held_out_ood_rows = image_inventory(OOD_INPUT, 'ood', paths=ood_paths)
if len(raw_rows) != EXPECTED_RAW_IMAGE_COUNT:
    raise ValueError('Expected ' + str(EXPECTED_RAW_IMAGE_COUNT) + ' original mango files; counted ' + str(len(raw_rows)))
if not oe_raw_rows:
    raise ValueError('The OE folder contains no readable supported image files.')
if len(held_out_ood_rows) != EXPECTED_OOD_IMAGE_COUNT:
    raise ValueError('Expected 500 held-out OOD photos for this first run; counted ' + str(len(held_out_ood_rows)) + '. Upload the additional 300 fresh images before creating a new run.')
if len({row['sha256'] for row in held_out_ood_rows}) != len(held_out_ood_rows) or len({row['decodedRgbSha256'] for row in held_out_ood_rows}) != len(held_out_ood_rows):
    raise ValueError('The 500 held-out OOD images must be distinct exact identities. Replace exact copies with fresh images before training.')
save_json_once(RUN_DIR / 'mango_inventory.json', {'images': portable_rows(raw_rows)})
save_json_once(RUN_DIR / 'oe_raw_inventory.json', {'images': portable_rows(oe_raw_rows)})
save_json_once(RUN_DIR / 'held_out_ood_raw_inventory.json', {'images': portable_rows(held_out_ood_rows)})
save_contact_sheets(oe_raw_rows, RUN_DIR / 'oe_review_previews', 'V4.5.0 OE input review: verify unsupported scope and related captures')
print('Raw mango:', len(raw_rows), 'OE discovered:', len(oe_raw_rows), 'Held-out OOD:', len(held_out_ood_rows))
print('OE review sheets:', RUN_DIR / 'oe_review_previews')
print('Adding 300 fresh OOD photos means uploading 300 additional reviewed files; do not move OE training photos into OOD.')


## Step 5: Retain the 62 reviewed mango identities and group related photos

**What:** Remove only actual exact copies while retaining the 62 previously reviewed distinct photographs.

**Why:** Similar appearance is not proof of duplication. Related captures must stay in one split so their similarities do not leak into evaluation.

**How:** Match every literal reviewed SHA-256 and class label, favor protected/named originals when choosing an exact-copy representative, and conservatively connect perceptually similar images. Cross-label exact copies stop for correction. Review pair sheets; near similarities are retained rather than deleted.

**Expected result:** The fresh audit proves all 62 hashes survived. Any protected exact-pixel conflict stops explicitly; no earlier run files are loaded.


In [ ]:
REVIEWED_IMAGES = [
    {
        "label": "anthracnose",
        "currentName": "verified mango leaf disease 21.jpg",
        "historicalName": "verified mango leaf disease 21.jpg",
        "sha256": "a557157b1e887d6f8f7067efbaa97447074114b2967c3c2337656d1936b45102"
    },
    {
        "label": "anthracnose",
        "currentName": "verified mango leaf disease 20.jpg",
        "historicalName": "verified mango leaf disease 20.jpg",
        "sha256": "87dad59aa6422d959e3fcfffd35ddca9c918e858dd7b220f1e016ae09d741f6f"
    },
    {
        "label": "anthracnose",
        "currentName": "DSC_0119.JPG",
        "historicalName": "DSC_0119.JPG",
        "sha256": "01e6fd123c9ae054d2d904fef25f543279c6c74914d44a33cd9c7488ca2861c6"
    },
    {
        "label": "anthracnose",
        "currentName": "DSC_0488.JPG",
        "historicalName": "DSC_0488.JPG",
        "sha256": "ee9676472204c9cdd0830dce3be63a9d47e646ecb9d2c70126ddcdac063e4ecd"
    },
    {
        "label": "anthracnose",
        "currentName": "DSC_0970.JPG",
        "historicalName": "DSC_0970.JPG",
        "sha256": "8b5d0b0e2500ff762a773b326c6a0a3ab545e51e682a253d3776be738d61123c"
    },
    {
        "label": "anthracnose",
        "currentName": "Anthracnose on young leaves.jpeg",
        "historicalName": "Anthracnose on young leaves.jpeg",
        "sha256": "5c10b4851fd67db6b5a21677e725d124491633afea45186fbe2e2a6f9bd554ec"
    },
    {
        "label": "anthracnose",
        "currentName": "AnthracnoseonMangoLeaf.jpeg",
        "historicalName": "AnthracnoseonMangoLeaf.jpeg",
        "sha256": "52a0fa64a504139aa65e6cbf807ecae21be6f4933bc4c1d5e2d8797a54de7227"
    },
    {
        "label": "sooty_mold",
        "currentName": "IMG_20260219_135000.jpg",
        "historicalName": "IMG_20260219_135000.jpg",
        "sha256": "2a22ee7f89ea03ad9ee924b5815ae71576d6ccd3305576dccc6f3bf00e31fce3"
    },
    {
        "label": "sooty_mold",
        "currentName": "viber_image_2024-11-15_13-17-51-567.jpg",
        "historicalName": "viber_image_2024-11-15_13-17-51-567.jpg",
        "sha256": "d785e72ca8291a194d5f56f2c75d2087ca3fd9728549246b888a3b964ce5b3c9"
    },
    {
        "label": "sooty_mold",
        "currentName": "IMG_20260219_134935.jpg",
        "historicalName": "IMG_20260219_134935.jpg",
        "sha256": "6c1c6c90f2dbfdb3ab1d451b90a9ce2769222aaa237bb820bf55f782e63498d5"
    },
    {
        "label": "sooty_mold",
        "currentName": "DSC_1029.JPG",
        "historicalName": "DSC_1029.JPG",
        "sha256": "7dfd201d72ea78a4557fdc5c5bddb594af538ff0270e020e73c9f382b3fa6f74"
    },
    {
        "label": "sooty_mold",
        "currentName": "724869371_27534362222843583_7670640125862653147_n.jpg",
        "historicalName": "724869371_27534362222843583_7670640125862653147_n.jpg",
        "sha256": "81c5a02299de1f2d66627a0f709a4f0c3235066749f7f8b203840b86f8a93b3c"
    },
    {
        "label": "sooty_mold",
        "currentName": "45a36510-0844-4baa-9795-3de7e0236896.jpg",
        "historicalName": "45a36510-0844-4baa-9795-3de7e0236896.jpg",
        "sha256": "32d90fe257d7ef563ef8166768b638a92a95c8196d33de90ced349c22b62a410"
    },
    {
        "label": "healthy_leaf",
        "currentName": "HealthyLeaf_2.jpg",
        "historicalName": "HealthyLeaf_2.jpg",
        "sha256": "e2e68e23f0fd8f88b999c0740cb577e126e78de61fa1fe71772d908ea2fc6375"
    },
    {
        "label": "healthy_leaf",
        "currentName": "HealthyLeaf_1.jpg",
        "historicalName": "HealthyLeaf_1.jpg",
        "sha256": "90a7907f7a259e404f6bb8c6558bb1081f2d2942249ebb5cac4b747a2f989e0c"
    },
    {
        "label": "mango_scab",
        "currentName": "15289a55-e368-4f61-a25c-85b2468243fd.jpg",
        "historicalName": "15289a55-e368-4f61-a25c-85b2468243fd.jpg",
        "sha256": "5a10c88561c498889b7668319c36fa1a432b942c21832e65d8c4a3063dbcf78b"
    },
    {
        "label": "mango_scab",
        "currentName": "DSC_0281.JPG",
        "historicalName": "DSC_0281.JPG",
        "sha256": "da1e429fc670c52fd75ba8cfe37b019606841ee7e012182f5696895e1ba13132"
    },
    {
        "label": "mango_scab",
        "currentName": "DSC_0966.JPG",
        "historicalName": "DSC_0966.JPG",
        "sha256": "de8cd5a96aefcccbd838a67411d44f13dcf9bf647200e40c71b7da28f26a1724"
    },
    {
        "label": "mango_scab",
        "currentName": "Scab.JPG",
        "historicalName": "Scab.JPG",
        "sha256": "10344e2e95ceeafab7d77e3e28e4ebbdebd699ec7e862c9c011516038e821068"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 70.jpg",
        "historicalName": "non verified mango disease 70.jpg",
        "sha256": "47a8192ddf400d3ba6397ba7d3acb21142d75267b80abe3911ba0d76e85ce995"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 71.jpg",
        "historicalName": "non verified mango disease 71.jpg",
        "sha256": "0c46b7f3c43190fb6e5663d6248f81e1eca31287ea8ab8cf3c2c9113aa46a848"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 73.jpg",
        "historicalName": "non verified mango disease 73.jpg",
        "sha256": "7562eee067e4a968c8f7c1f3281125bcb3546dcab2795f196ee11286e836ac09"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 68.jpg",
        "historicalName": "non verified mango disease 68.jpg",
        "sha256": "281064014381e3f9d4cd163b62ec6e643364ee5a3a0840875f9c310912a30c8d"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 67.jpg",
        "historicalName": "non verified mango disease 67.jpg",
        "sha256": "6e99067feab48b78d0e0c4260e342c1b64673dab00a5c18099310bb9053f1dd7"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 66.jpg",
        "historicalName": "non verified mango disease 66.jpg",
        "sha256": "0e515243d9cec0d0e1f970d4ba79081606c2280ee02176e30fa24b2817784cda"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 65.jpg",
        "historicalName": "non verified mango disease 65.jpg",
        "sha256": "ae4bfcc5cd3eba0f652aa4355d1780ef6efe27af71f3d020a15b1eee2c0f975b"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 56.jpg",
        "historicalName": "non verified mango disease 56.jpg",
        "sha256": "ad4415243dc86b38c7ac7f529fdb2e09f7fc9410ded477a553a0f33ef02736fa"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 53.jpg",
        "historicalName": "non verified mango disease 53.jpg",
        "sha256": "89652a16e8c0590a4fd51b9ff66c7ff9fb95a7ded8aece63c85565dad39f5fff"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 52.jpg",
        "historicalName": "non verified mango disease 52.jpg",
        "sha256": "cfb52caae3133082bc5f70c82f4b4c73589a65087409383e87453d405648391e"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 51.jpg",
        "historicalName": "non verified mango disease 51.jpg",
        "sha256": "e2c7e67b3026efe9eac0941272a41654f54c8317a2d11d7ba0eb81699891a2bd"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 49.jpg",
        "historicalName": "non verified mango disease 49.jpg",
        "sha256": "7b87cb61eb26ffffde585b8cbb254795f0da834ce1c306bae76fe2b6cdf0f736"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 11.jpg",
        "historicalName": "non verified mango disease 11.jpg",
        "sha256": "81a2f9abde6762ed891db43c570a5339d465682e3817c4f79917f8f497ed77cd"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 10.jpg",
        "historicalName": "verified mango disease 10.jpg",
        "sha256": "afcb278a84a3abf145a2d396bbcfa99b5818b217764c3fef07246d6993421095"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 9.jpg",
        "historicalName": "verified mango disease 9.jpg",
        "sha256": "6fdc1ff1319c85198ae2d42978e9c3ec11ee82b754e982915a3df896e922e187"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 1.jpg",
        "historicalName": "verified mango disease 1.jpg",
        "sha256": "156d089cee4c22e7256a0aa6ef48c84025507fcafda3b463fe39f9d9f852cd87"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 58.jpg",
        "historicalName": "verified mango disease 58.jpg",
        "sha256": "7fe031c22293d0c980ddf2744d2b3732bcbe22c659281c67ef7f24ce577619c6"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 36.jpg",
        "historicalName": "verified mango disease 36.jpg",
        "sha256": "22deb301d133db0af636c788508379203b6b90e4ca3ce9aa3ce238069cccfc4e"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 32.jpg",
        "historicalName": "verified mango disease 32.jpg",
        "sha256": "bbfbb925ba9bc8cb2595d501ab8180bc7378dfb199a1110793743e79a40e376f"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 6.jpg",
        "historicalName": "verified mango disease 6.jpg",
        "sha256": "262ea4d6d82594a4333470180b3de1a77560fafc3acc1e0a75144260e1928e15"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 2.jpg",
        "historicalName": "verified mango disease 2.jpg",
        "sha256": "1da80fe4b68725c7c39d78af8e4a42c65316c4e29d2be3756bef74c02a51d1d0"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 63.jpg",
        "historicalName": "verified mango disease 63.jpg",
        "sha256": "95e7184c9c0c2c0f5e729d16513410470c77e07bc4a787d50c06f79a55977cd8"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 57.jpg",
        "historicalName": "verified mango disease 57.jpg",
        "sha256": "9c9bc581355a1a4bfcd0befe2051135b7903d219df9320587cd88a28c94db5f6"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 7.jpg",
        "historicalName": "verified mango disease 7.jpg",
        "sha256": "0a7326dd87f77caad7cae79d709bd7cdfcb5a0aec2ef96463782b2920b5cb55c"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 55.jpg",
        "historicalName": "verified mango disease 55.jpg",
        "sha256": "d70902b669201181ee27847a13e4bbf684a371c18957b2cbb20b6cfcbf75275a"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 45.jpg",
        "historicalName": "verified mango disease 45.jpg",
        "sha256": "0827d4061caffa787cb8fab501fb3f25f6e4ef40f434ee03787136b0c8f2d4b7"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 54.jpg",
        "historicalName": "verified mango disease 54.jpg",
        "sha256": "4eee0b91cfa3f2c5b66a0f334d2d2e487fd666d7aa10b3ea9f8bdfc9d36dbfea"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 8.jpg",
        "historicalName": "verified mango disease 8.jpg",
        "sha256": "00239798d21836b06afea3d213aff445d509eabab9cfa5515e687afc20447e97"
    },
    {
        "label": "mango_scab",
        "currentName": "DSC_1014.JPG",
        "historicalName": "DSC_1014.JPG",
        "sha256": "348841118bd2fed6966a5837ddd7dcde55cb053057371a39ce4f051aa5627bdf"
    },
    {
        "label": "mango_scab",
        "currentName": "Scab1.JPG",
        "historicalName": "Scab1.JPG",
        "sha256": "dbf4ee9f20018da3f04c542d9b369403bb8fbf0149b36c14a4363782d68d174e"
    },
    {
        "label": "mango_scab",
        "currentName": "725146731_2449764938863852_7559841505625140608_n.jpg",
        "historicalName": "725146731_2449764938863852_7559841505625140608_n.jpg",
        "sha256": "a86e5a775f479a63fe2528305bc350300435f8b048494a06ae13e1fd78d0d0f5"
    },
    {
        "label": "healthy_fruit",
        "currentName": "DSC_1021.JPG",
        "historicalName": "DSC_1021.JPG",
        "sha256": "9872562f0d3dfe137e0db5edfbaeaa602aef7444aacded8d18ce9de1bfbf5390"
    },
    {
        "label": "healthy_fruit",
        "currentName": "DSC_1019.JPG",
        "historicalName": "DSC_1019.JPG",
        "sha256": "f744b6410bd2ba2897372201bd1179144fbcb6860e11cf2ba083e1fc8e4f1d9e"
    },
    {
        "label": "healthy_fruit",
        "currentName": "DSC_0436.JPG",
        "historicalName": "DSC_0436.JPG",
        "sha256": "7053f37a4c34867f2c77d44ca764def21315667a24ce4cf388586a6be866d458"
    },
    {
        "label": "healthy_fruit",
        "currentName": "DSC_0205.JPG",
        "historicalName": "DSC_0205.JPG",
        "sha256": "0a1ab9392878b940fc828aa7e7659b0ee2897244935f0dd57e8e012384c9cc53"
    },
    {
        "label": "healthy_fruit",
        "currentName": "DSC_0204.JPG",
        "historicalName": "DSC_0204.JPG",
        "sha256": "5e832bc5a93f67dda12f79405fa39e89f7208052bb5ee6e627711967c6ee668c"
    },
    {
        "label": "healthy_fruit",
        "currentName": "DSC_0201 (1).JPG",
        "historicalName": "DSC_0201 (1).JPG",
        "sha256": "84e0237792a86d60a92e12a4da910e7bfe20f60439aff39f3534d6eaeb1c35f4"
    },
    {
        "label": "healthy_fruit",
        "currentName": "727799974_830567853245491_1446073817623999359_n.jpg",
        "historicalName": "727799974_830567853245491_1446073817623999359_n.jpg",
        "sha256": "9e0436c4d16f0517ffb395477f0567ff046fd41d04311ae74c1abb224c496504"
    },
    {
        "label": "healthy_fruit",
        "currentName": "725318273_1696519271391031_8474708639771176175_n.jpg",
        "historicalName": "725318273_1696519271391031_8474708639771176175_n.jpg",
        "sha256": "18c047f48c6deb7b423919369488485c608300cbba79d078428954694073ba4f"
    },
    {
        "label": "healthy_fruit",
        "currentName": "723960758_2381115535716542_3913785565978577803_n.jpg",
        "historicalName": "723960758_2381115535716542_3913785565978577803_n.jpg",
        "sha256": "3fb605dbbc492f8466267cbb8ec52ceede4455c676dc54a66db37b7e909ed644"
    },
    {
        "label": "healthy_fruit",
        "currentName": "723624552_976005712089787_680463173797746853_n.jpg",
        "historicalName": "723624552_976005712089787_680463173797746853_n.jpg",
        "sha256": "bccae50356a84611268b9fd0b5f5bf5517b222ac223d298084b121e169550b64"
    },
    {
        "label": "healthy_fruit",
        "currentName": "723624552_976005712089787_680463173797746853_n_2.jpg",
        "historicalName": "723624552_976005712089787_680463173797746853_n_2.jpg",
        "sha256": "b1c8ddc707402b023f61be78b900eb878af012c14964e1dbf25b7aab060c2310"
    },
    {
        "label": "healthy_fruit",
        "currentName": "719671069_1291039922786074_6951448894192280220_n.jpg",
        "historicalName": "719671069_1291039922786074_6951448894192280220_n.jpg",
        "sha256": "61cabb3193580b025925fe0177dfdf9a62cf7ea19a18df33218c41842349439c"
    }
]


In [ ]:
# A failed Step 4 must not be bypassed by clicking the next cell.
require_completed_inventories()

GRADCAM_NAMES = {
    'anthracnose': 'DSC_0488.JPG',
    'sooty_mold': 'IMG_20260219_134935.jpg',
    'healthy_leaf': 'HealthyLeaf_1.jpg',
    'mango_scab': 'verified mango disease 67.jpg',
    'healthy_fruit': '723624552_976005712089787_680463173797746853_n.jpg',
}
reviewed_hashes = {row['sha256'] for row in REVIEWED_IMAGES}
raw_by_hash = defaultdict(list)
for row in raw_rows:
    raw_by_hash[row['sha256']].append(row)
missing_reviewed = [row for row in REVIEWED_IMAGES if row['sha256'] not in raw_by_hash]
wrong_labels = [row for row in REVIEWED_IMAGES if any(raw['label'] != row['label'] for raw in raw_by_hash[row['sha256']])]
if missing_reviewed or wrong_labels:
    raise RuntimeError('A protected reviewed image is missing, relabeled, or changed; resolve inventory before continuing.')
save_json_once(RUN_DIR / 'reviewed_images.json', REVIEWED_IMAGES)


def exact_representatives(rows, protected_hashes=None, preferred_names=None):
    protected_hashes = protected_hashes or set()
    preferred_names = preferred_names or {}
    decoded = defaultdict(list)
    for row in rows:
        decoded[row['decodedRgbSha256']].append(row)
    retained = []
    audit = []
    for fingerprint, group in sorted(decoded.items()):
        labels = {row.get('label') for row in group}
        if len(labels) != 1:
            raise RuntimeError('Exact decoded image has conflicting class labels: ' + str([row['sourceRelativePath'] for row in group]))
        protected = {row['sha256'] for row in group if row['sha256'] in protected_hashes}
        if len(protected) > 1:
            raise RuntimeError('Two protected distinct identities have identical decoded pixels; explicit conflict resolution required.')
        chosen = min(group, key=lambda row: (
            row['sha256'] not in protected_hashes,
            Path(row['sourceRelativePath']).name != preferred_names.get(row.get('label')),
            row['sourceRelativePath'],
        ))
        retained.append(dict(chosen))
        if len(group) > 1:
            audit.append({'keptId': chosen['id'], 'copies': portable_rows(group)})
    if not protected_hashes.issubset({row['sha256'] for row in retained}):
        raise RuntimeError('Exact-copy reduction would remove a protected reviewed fingerprint.')
    return retained, audit


def near_pairs(rows):
    pairs = []
    for index, first in enumerate(rows):
        first_hash = int(first['phash'], 16)
        for second in rows[index + 1:]:
            distance = (first_hash ^ int(second['phash'], 16)).bit_count()
            if distance <= PHASH_DISTANCE:
                pairs.append({'firstId': first['id'], 'secondId': second['id'], 'distance': distance})
    return pairs


def connected_groups(rows, pairs, distinct_pairs, extra_groups):
    parents = {row['id']: row['id'] for row in rows}

    def root(key):
        while parents[key] != key:
            parents[key] = parents[parents[key]]
            key = parents[key]
        return key

    def join(left, right):
        first, second = root(left), root(right)
        parents[max(first, second)] = min(first, second)

    audited = {tuple(sorted((pair['firstId'], pair['secondId']))) for pair in pairs}
    distinct = {tuple(sorted(pair)) for pair in distinct_pairs}
    if any(len(pair) != 2 for pair in distinct_pairs) or not distinct.issubset(audited):
        raise ValueError('Distinct decisions must name complete audited pairs.')
    for pair in audited - distinct:
        join(*pair)
    for group in extra_groups:
        if len(group) < 2 or any(key not in parents for key in group):
            raise ValueError('Related-capture groups require at least two known image IDs.')
        for key in group[1:]:
            join(group[0], key)
    return {key: root(key) for key in parents}


retained_rows, exact_groups = exact_representatives(raw_rows, reviewed_hashes, GRADCAM_NAMES)
similarity_pairs = near_pairs(retained_rows)
save_json_once(RUN_DIR / 'duplicate_audit.json', {
    'rawCount': len(raw_rows), 'retainedCount': len(retained_rows),
    'protectedCount': len(reviewed_hashes), 'exactCopyGroups': exact_groups,
    'similarityDistance': PHASH_DISTANCE, 'similarityPairs': similarity_pairs,
    'policy': 'Exact copies reduced; all 62 protected hashes retained; near similarities retained and grouped.',
})
flagged_ids = {pair[key] for pair in similarity_pairs for key in ('firstId', 'secondId')}
save_contact_sheets([row for row in retained_rows if row['id'] in flagged_ids], RUN_DIR / 'duplicate_previews', 'V4.5.0 retained mango similarities: inspect pair audit')
save_contact_sheets(retained_rows, RUN_DIR / 'retained_previews', 'V4.5.0 retained mango images')
print('Retained:', len(retained_rows), 'Protected fingerprints:', len(reviewed_hashes), 'Near pairs:', len(similarity_pairs))


### Approve the mango grouping review

Read `duplicate_audit.json`, `duplicate_previews/`, and acquisition notes. Perceptual matches remain grouped by default. Only list a pair in `DISTINCT_SIMILAR_PAIRS` if you inspected both and can justify independent captures. Use `ADDITIONAL_RELATED_GROUPS` for the same specimen/tree/session even when perceptual hashing missed the relationship. Images from different classes can share a whole group; label conflict must be inspected, never split silently.


In [ ]:
DUPLICATE_REVIEW_COMPLETE = False
DISTINCT_SIMILAR_PAIRS = []
ADDITIONAL_RELATED_GROUPS = []
DUPLICATE_REVIEW_NOTES = ''
MANGO_SOURCE_NOTES = {}
MANGO_CAPTURE_GROUPS = {}  # Image ID -> globally meaningful specimen/session name, when known.
MANGO_SOURCE_GROUPS = {}  # Image ID -> original source-image/rotation sequence identity, when known.

if not DUPLICATE_REVIEW_COMPLETE or not DUPLICATE_REVIEW_NOTES.strip():
    raise RuntimeError('Inspect all mango similarity sheets and related-capture notes; complete the review settings above.')
known_mango_ids = {row['id'] for row in retained_rows}
if not set(MANGO_CAPTURE_GROUPS).issubset(known_mango_ids) or not set(MANGO_SOURCE_GROUPS).issubset(known_mango_ids):
    raise ValueError('Mango capture/source group keys must be known retained IDs.')
metadata_groups = []
for mapping in (MANGO_CAPTURE_GROUPS, MANGO_SOURCE_GROUPS):
    grouped = defaultdict(list)
    for key, value in mapping.items():
        if str(value).strip():
            grouped[str(value).strip()].append(key)
    metadata_groups.extend(ids for ids in grouped.values() if len(ids) > 1)
group_assignments = connected_groups(retained_rows, similarity_pairs, DISTINCT_SIMILAR_PAIRS, ADDITIONAL_RELATED_GROUPS + metadata_groups)
for row in retained_rows:
    row['groupId'] = group_assignments[row['id']]
    row['captureGroup'] = str(MANGO_CAPTURE_GROUPS.get(row['id'], '')).strip()
    row['sourceGroup'] = str(MANGO_SOURCE_GROUPS.get(row['id'], '')).strip()
save_json_once(RUN_DIR / 'duplicate_review.json', {
    'complete': True, 'distinctSimilarPairs': DISTINCT_SIMILAR_PAIRS,
    'additionalRelatedGroups': ADDITIONAL_RELATED_GROUPS,
    'notes': DUPLICATE_REVIEW_NOTES, 'sourceNotes': MANGO_SOURCE_NOTES,
    'captureGroups': MANGO_CAPTURE_GROUPS, 'sourceGroups': MANGO_SOURCE_GROUPS,
    'groupAssignments': group_assignments,
})
print('All 62 reviewed fingerprints retained; complete related groups:', len(set(group_assignments.values())))


## Step 6: Review OE scope, provenance, related captures, and leakage

**What:** Choose unsupported OE photos and inspect overlap with every mango and held-out OOD identity.

**Why:** Training on valid mango photos as outliers contradicts supervised training. Using OOD reporting photos for OE would contaminate evaluation.

**How:** Exact-copy OE representatives are counted automatically. Review all contact sheets. An explicit approve-all option is available only after your visual review; source notes describe what is actually known. Group related OE captures. Exact cross-pool matches stop; perceptual cross-pool flags need explicit distinct-image decisions and notes.

**Expected result:** Approved OE identities, grouping decisions, source notes, and a leakage receipt are saved. Unknown counts are reported, not assumed.


In [ ]:
oe_rows, oe_exact_groups = exact_representatives(oe_raw_rows)
oe_similarity_pairs = near_pairs(oe_rows)
save_json_once(RUN_DIR / 'oe_duplicate_audit.json', {
    'rawCount': len(oe_raw_rows), 'retainedCount': len(oe_rows),
    'exactCopyGroups': oe_exact_groups, 'similarityPairs': oe_similarity_pairs,
})
save_contact_sheets(oe_rows, RUN_DIR / 'oe_retained_previews', 'V4.5.0 OE representative review')
print('OE raw:', len(oe_raw_rows), 'Exact representatives:', len(oe_rows), 'Near pairs:', len(oe_similarity_pairs))


In [ ]:
OE_REVIEW_COMPLETE = False
APPROVE_ALL_OE_AS_UNSUPPORTED = False  # True only after reviewing every retained OE photo.
APPROVED_OE_IDS = []  # Use selected identities instead of approve-all when needed.
REJECTED_OE_IDS = []
OE_DISTINCT_SIMILAR_PAIRS = []
OE_ADDITIONAL_RELATED_GROUPS = []
OE_SOURCE_NOTES = ''  # Honest collection origin; do not invent per-photo licenses or capture devices.
OE_SOURCE_NOTES_BY_ID = {}
OE_REVIEW_NOTES = ''
OE_CAPTURE_GROUPS = {}  # ID -> known same specimen/session, e.g. phone-owner/date/session.
OE_SOURCE_GROUPS = {}  # ID -> known original image/burst, not an entire dataset or website.
HELD_OUT_OOD_CAPTURE_GROUPS = {}  # Use the same global naming scheme as mango/OE.
HELD_OUT_OOD_SOURCE_GROUPS = {}
HELD_OUT_OOD_GROUP_REVIEW_COMPLETE = False
HELD_OUT_OOD_GROUP_REVIEW_NOTES = ''

known_oe_ids = {row['id'] for row in oe_rows}
if not OE_REVIEW_COMPLETE or not OE_REVIEW_NOTES.strip() or not OE_SOURCE_NOTES.strip():
    raise RuntimeError('Inspect OE sheets, confirm unsupported scope and related captures, then enter honest notes and approval.')
if not set(APPROVED_OE_IDS).issubset(known_oe_ids) or not set(REJECTED_OE_IDS).issubset(known_oe_ids):
    raise ValueError('OE approvals/rejections must be known retained IDs.')
if len(APPROVED_OE_IDS) != len(set(APPROVED_OE_IDS)) or len(REJECTED_OE_IDS) != len(set(REJECTED_OE_IDS)):
    raise ValueError('OE decision lists must not repeat identities.')
if set(APPROVED_OE_IDS) & set(REJECTED_OE_IDS):
    raise ValueError('An OE identity cannot be both approved and rejected.')
if APPROVE_ALL_OE_AS_UNSUPPORTED and APPROVED_OE_IDS:
    raise ValueError('Choose approve-all or a selected-ID list, not both.')
approved_oe_ids = (known_oe_ids if APPROVE_ALL_OE_AS_UNSUPPORTED else set(APPROVED_OE_IDS)) - set(REJECTED_OE_IDS)
if not approved_oe_ids:
    raise ValueError('OE training needs at least one visually reviewed unsupported identity.')
if not HELD_OUT_OOD_GROUP_REVIEW_COMPLETE or not HELD_OUT_OOD_GROUP_REVIEW_NOTES.strip():
    raise RuntimeError('Review held-out OOD acquisition/source groups before training; record known related photos and honest notes.')
known_ood_ids = {row['id'] for row in held_out_ood_rows}
for mapping, known in ((OE_CAPTURE_GROUPS, known_oe_ids), (OE_SOURCE_GROUPS, known_oe_ids),
                       (HELD_OUT_OOD_CAPTURE_GROUPS, known_ood_ids), (HELD_OUT_OOD_SOURCE_GROUPS, known_ood_ids)):
    if not set(mapping).issubset(known):
        raise ValueError('Capture/source metadata uses an unknown image ID.')
metadata_groups = []
for mapping in (OE_CAPTURE_GROUPS, OE_SOURCE_GROUPS):
    grouped = defaultdict(list)
    for key, value in mapping.items():
        if str(value).strip():
            grouped[str(value).strip()].append(key)
    metadata_groups.extend(ids for ids in grouped.values() if len(ids) > 1)
oe_group_assignments = connected_groups(oe_rows, oe_similarity_pairs, OE_DISTINCT_SIMILAR_PAIRS, OE_ADDITIONAL_RELATED_GROUPS + metadata_groups)
for row in oe_rows:
    row['groupId'] = oe_group_assignments[row['id']]
    row['captureGroup'] = str(OE_CAPTURE_GROUPS.get(row['id'], '')).strip()
    row['sourceGroup'] = str(OE_SOURCE_GROUPS.get(row['id'], '')).strip()
for row in held_out_ood_rows:
    row['captureGroup'] = str(HELD_OUT_OOD_CAPTURE_GROUPS.get(row['id'], '')).strip()
    row['sourceGroup'] = str(HELD_OUT_OOD_SOURCE_GROUPS.get(row['id'], '')).strip()
approved_oe_rows = [dict(row) for row in oe_rows if row['id'] in approved_oe_ids]
save_json_once(RUN_DIR / 'oe_review.json', {
    'complete': True, 'approvedIds': sorted(approved_oe_ids), 'rejectedIds': REJECTED_OE_IDS,
    'distinctSimilarPairs': OE_DISTINCT_SIMILAR_PAIRS,
    'additionalRelatedGroups': OE_ADDITIONAL_RELATED_GROUPS,
    'sourceNotes': OE_SOURCE_NOTES, 'sourceNotesById': OE_SOURCE_NOTES_BY_ID,
    'notes': OE_REVIEW_NOTES, 'groupAssignments': oe_group_assignments,
    'captureGroups': OE_CAPTURE_GROUPS, 'sourceGroups': OE_SOURCE_GROUPS,
})
save_json_once(RUN_DIR / 'oe_inventory.json', {'images': portable_rows(oe_rows)})
save_json_once(RUN_DIR / 'held_out_ood_inventory.json', {'images': portable_rows(held_out_ood_rows)})
save_json_once(RUN_DIR / 'held_out_ood_group_review.json', {
    'complete': True, 'notes': HELD_OUT_OOD_GROUP_REVIEW_NOTES,
    'captureGroups': HELD_OUT_OOD_CAPTURE_GROUPS, 'sourceGroups': HELD_OUT_OOD_SOURCE_GROUPS,
})


def cross_pool_flags(first_rows, second_rows):
    flags = []
    for first in first_rows:
        for second in second_rows:
            exact = first['sha256'] == second['sha256'] or first['decodedRgbSha256'] == second['decodedRgbSha256']
            metadata_overlap = any(first.get(field) and first.get(field) == second.get(field)
                                   for field in ('captureGroup', 'sourceGroup'))
            distance = (int(first['phash'], 16) ^ int(second['phash'], 16)).bit_count()
            if exact or metadata_overlap or distance <= PHASH_DISTANCE:
                key = first['pool'] + ':' + first['id'] + '|' + second['pool'] + ':' + second['id']
                flags.append({
                    'key': key, 'firstId': first['id'], 'secondId': second['id'],
                    'firstPool': first['pool'], 'secondPool': second['pool'],
                    'firstPath': first['sourceRelativePath'], 'secondPath': second['sourceRelativePath'],
                    'exactMatch': exact, 'sourceOrCaptureOverlap': bool(metadata_overlap), 'distance': distance,
                })
    return flags


cross_flags = cross_pool_flags(approved_oe_rows, retained_rows) + cross_pool_flags(approved_oe_rows, held_out_ood_rows)
# Mango/OOD overlap is also unacceptable, regardless of whether an OE row matched.
cross_flags += cross_pool_flags(retained_rows, held_out_ood_rows)
save_json_once(RUN_DIR / 'cross_pool_overlap_audit.json', {'phashDistance': PHASH_DISTANCE, 'pairs': cross_flags})
print('Cross-pool flags:', len(cross_flags), 'Audit:', RUN_DIR / 'cross_pool_overlap_audit.json')
if any(row['exactMatch'] or row['sourceOrCaptureOverlap'] for row in cross_flags):
    raise RuntimeError('An exact image or known source/capture group crosses mango/OE/held-out OOD pools. Correct source collections and create a new run; do not train on evaluation photos.')


### Approve distinct cross-pool similarities

For each non-exact flag, inspect both original photos using its paths. A coincidental similarity may be recorded as `{"decision": "distinct", "notes": "Specific reason these are independent photos."}` under its complete `key`. If two photos are related captures, remove the training/evaluation overlap from the original collections and create a new run. No flags means the decisions dictionary remains empty. Hash checks cannot detect every related capture; acquisition grouping review still matters.


In [ ]:
CROSS_POOL_DISTINCT_DECISIONS = {}
if set(CROSS_POOL_DISTINCT_DECISIONS) != {row['key'] for row in cross_flags}:
    raise RuntimeError('Every cross-pool similarity needs a distinct decision and notes in CROSS_POOL_DISTINCT_DECISIONS.')
for flag in cross_flags:
    decision = CROSS_POOL_DISTINCT_DECISIONS[flag['key']]
    if decision.get('decision') != 'distinct' or not str(decision.get('notes', '')).strip():
        raise ValueError('Unresolved cross-pool similarity: ' + flag['key'])
save_json_once(RUN_DIR / 'cross_pool_overlap_review.json', {
    'complete': True, 'decisions': CROSS_POOL_DISTINCT_DECISIONS,
    'exactMatchCount': 0, 'oodPredictionsRun': False,
})
save_json_once(RUN_DIR / 'oe_training_manifest.json', {
    'images': portable_rows(approved_oe_rows), 'rawCount': len(oe_raw_rows),
    'approvedCount': len(approved_oe_rows), 'usage': 'OE training only; never threshold tuning or reporting',
})
print('Approved OE training photos:', len(approved_oe_rows))


## Step 7: Create a fresh grouped 70/15/15 split and reserve preview photos

**What:** Assign whole related mango groups to training, validation, or test and copy original image bytes into this run.

**Why:** Fresh group-level allocation prevents related images crossing splits. The same seven original photos must illustrate augmentation and Grad-CAM without becoming test-selection examples.

**How:** Reserve the five exact named originals and their complete groups in validation. Seed-select the other two classes before any predictions and reserve their whole groups too. Target 70/15/15 per class, print actual counts, and require review before sealing copies and the split zip.

**Expected result:** A new input manifest, split archive, and seven preview identities. Group constraints can make actual fractions differ from exact targets.


In [ ]:
groups = defaultdict(list)
for row in retained_rows:
    groups[row['groupId']].append(row)
named_examples = {}
for label, filename in GRADCAM_NAMES.items():
    matches = [row for row in raw_rows if row['label'] == label and Path(row['sourceRelativePath']).name == filename]
    if len(matches) != 1:
        raise ValueError('The requested original preview filename is missing or ambiguous: ' + filename)
    match = matches[0]
    chosen = next((row for row in retained_rows if row['sha256'] == match['sha256']), None)
    if chosen is None:
        raise RuntimeError('Requested original preview identity was not retained: ' + filename)
    named_examples[label] = chosen
preview_rng = random.Random(SEED)
for label in CLASS_NAMES:
    if label not in named_examples:
        choices = sorted((row for row in retained_rows if row['label'] == label), key=lambda row: row['id'])
        if not choices:
            raise ValueError('No retained photos for preview class: ' + label)
        named_examples[label] = preview_rng.choice(choices)
forced_validation = {row['groupId'] for row in named_examples.values()}


def assign_mango_groups(groups, class_names, reserved_group_ids, seed):
    subset_names = ('train', 'val', 'test')
    totals = np.array([sum(row['label'] == label for group in groups.values() for row in group)
                       for label in class_names], dtype=float)
    targets = np.array([0.70, 0.15, 0.15])[:, None] * totals[None, :]
    counts = np.zeros_like(targets)
    assignments = {}

    def contribution(key):
        return np.array([sum(row['label'] == label for row in groups[key]) for label in class_names], dtype=float)

    if not set(reserved_group_ids).issubset(groups):
        raise ValueError('Reserved preview groups must exist.')
    for key in sorted(reserved_group_ids):
        assignments[key] = 'val'
        counts[1] += contribution(key)
    remaining = sorted(key for key in groups if key not in assignments)
    random.Random(seed).shuffle(remaining)
    remaining.sort(key=lambda key: -len(groups[key]))
    for key in remaining:
        values = contribution(key)
        scores = []
        for index in range(3):
            candidate = counts.copy()
            candidate[index] += values
            scores.append(float(np.sum(((candidate - targets) / np.maximum(targets, 1)) ** 2)))
        winner = int(np.argmin(scores))
        assignments[key] = subset_names[winner]
        counts[winner] += values
    if np.any(counts == 0):
        raise RuntimeError('A class is missing from a split. Review oversized groups, never split related captures.')
    return assignments, counts


assignments, split_counts = assign_mango_groups(groups, CLASS_NAMES, forced_validation, SEED)
records = []
for row in sorted(retained_rows, key=lambda item: (CLASS_NAMES.index(item['label']), item['id'])):
    saved = dict(row)
    saved['subset'] = assignments[row['groupId']]
    saved['relativePath'] = saved['subset'] + '/' + saved['label'] + '/' + saved['id'] + '__' + Path(saved['sourceRelativePath']).name
    records.append(saved)
counts_by_subset = {
    subset: {label: sum(row['subset'] == subset and row['label'] == label for row in records) for label in CLASS_NAMES}
    for subset in ('train', 'val', 'test')
}
print('Proposed split counts:', json.dumps(counts_by_subset, indent=2))
print('Actual fractions:', {subset: sum(row['subset'] == subset for row in records) / len(records) for subset in counts_by_subset})
print('Preview originals:', {label: row['sourceRelativePath'] for label, row in named_examples.items()})


In [ ]:
SPLIT_REVIEW_COMPLETE = False
SPLIT_REVIEW_NOTES = ''
if not SPLIT_REVIEW_COMPLETE or not SPLIT_REVIEW_NOTES.strip():
    raise RuntimeError('Review printed counts, complete-group assignments, and the reserved original previews before approving.')
verify_image_rows(raw_rows)
verify_image_rows(approved_oe_rows)
verify_image_rows(held_out_ood_rows)
if DATASET_DIR.exists():
    raise RuntimeError('Dataset copies already exist. Do not recreate or overwrite the split in this run.')
DATASET_DIR.mkdir()
for row in records:
    destination = safe_child(DATASET_DIR, row['relativePath'])
    destination.parent.mkdir(parents=True, exist_ok=True)
    with Path(row['path']).open('rb') as source, destination.open('xb') as target:
        shutil.copyfileobj(source, target)
    row['path'] = str(destination)
verify_image_rows(records)

# Preserve OE training originals inside the run so later threshold checks can
# inspect the actual pixels even when the mounted raw input location changes.
oe_dataset = RUN_DIR / 'oe_dataset'
oe_dataset.mkdir()
for row in approved_oe_rows:
    filename = row['id'] + '__' + Path(row['sourceRelativePath']).name
    row['relativePath'] = 'oe_dataset/' + filename
    destination = safe_child(RUN_DIR, row['relativePath'])
    with Path(row['path']).open('rb') as source, destination.open('xb') as target:
        shutil.copyfileobj(source, target)
    row['path'] = str(destination)
verify_image_rows(approved_oe_rows)
save_json_once(RUN_DIR / 'oe_dataset_manifest.json', {'images': portable_rows(approved_oe_rows)})
train_rows = [row for row in records if row['subset'] == 'train']
val_rows = [row for row in records if row['subset'] == 'val']
by_id = {row['id']: row for row in records}
example_rows = [by_id[named_examples[label]['id']] for label in CLASS_NAMES]
if any(row['subset'] != 'val' for row in example_rows):
    raise RuntimeError('All preview identities and their complete groups must remain in validation.')
manifest = {
    'schemaVersion': 1, 'experimentVersion': EXPERIMENT_VERSION, 'classNames': CLASS_NAMES,
    'counts': counts_by_subset, 'splitFractionsTarget': [0.70, 0.15, 0.15],
    'splitPolicy': 'Fresh grouped allocation; all seven preview groups reserved in validation before predictions',
    'images': portable_rows(records), 'validationProbabilityOrder': 'images subset=val in listed order',
}
save_json_once(RUN_DIR / 'input_manifest.json', manifest)
save_json_once(RUN_DIR / 'class_names.json', CLASS_NAMES)
save_json_once(RUN_DIR / 'split_review.json', {'complete': True, 'notes': SPLIT_REVIEW_NOTES, 'counts': counts_by_subset})
save_json_once(RUN_DIR / 'example_selection.json', {
    'seed': SEED, 'subset': 'val', 'images': portable_rows(example_rows),
    'selectionBeforePredictions': True, 'namedOriginalFilenames': GRADCAM_NAMES,
})
with zipfile.ZipFile(RUN_DIR / 'dataset_split.zip', 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for row in records:
        archive.write(row['path'], row['relativePath'])
print('Frozen split copies:', DATASET_DIR)


## Step 8: Define six training augmentations and inspect real previews

**What:** Use flip, rotation, zoom, shear, global brightness, and random contrast 0.1.

**Why:** These are candidate variations, not a proven optimum. Visible disease signs and valid specimen boundaries should survive combined transforms.

**How:** Apply transformations only to training batches. Preview each category and a combined sample on the same seven original validation photos without inserting preview copies into scoring. Brightness and contrast use the 0–255 range. Contrast factors range from 0.9 to 1.1.

**Expected result:** augmentation_preview.png has a v4.5.0 label, seven class rows, category headings, and a caption. Review it before training.


In [ ]:
class RandomShear(tf.keras.layers.Layer):
    def __init__(self, factor=0.2, **kwargs):
        super().__init__(**kwargs)
        self.factor = factor

    def call(self, inputs, training=False):
        if not training:
            return inputs
        shape = tf.shape(inputs)
        count = shape[0]
        center_x = tf.cast(shape[2], tf.float32) / 2.0
        center_y = tf.cast(shape[1], tf.float32) / 2.0
        shear_x = tf.random.uniform([count], -self.factor, self.factor)
        shear_y = tf.random.uniform([count], -self.factor, self.factor)
        one, zero = tf.ones_like(shear_x), tf.zeros_like(shear_x)
        transforms = tf.stack((one, shear_x, -shear_x * center_y, shear_y, one, -shear_y * center_x, zero, zero), axis=1)
        return tf.raw_ops.ImageProjectiveTransformV3(
            images=inputs, transforms=transforms, output_shape=shape[1:3],
            interpolation='BILINEAR', fill_mode='NEAREST', fill_value=0.0,
        )


def augmentation_layers(seed):
    return [
        tf.keras.layers.RandomFlip('horizontal_and_vertical', seed=seed),
        tf.keras.layers.RandomRotation(30 / 360, fill_mode='nearest', seed=seed + 1),
        tf.keras.layers.RandomZoom(0.2, fill_mode='nearest', seed=seed + 2),
        RandomShear(0.2),
        tf.keras.layers.RandomBrightness(0.2, value_range=(0, 255), seed=seed + 3),
        tf.keras.layers.RandomContrast(CONTRAST_FACTOR, value_range=(0, 255), seed=seed + 4),
    ]


def augment_batch(pixels, layers):
    output = pixels
    for layer in layers:
        output = layer(output, training=True)
    return tf.clip_by_value(output, 0.0, 255.0)


def image_dataset(rows, labelled=False):
    def examples():
        for row in rows:
            pixels = load_pixels(row['path']).numpy().astype(np.float32)
            if labelled:
                yield pixels, np.int32(CLASS_NAMES.index(row['label']))
            else:
                yield pixels
    signature = tf.TensorSpec((224, 224, 3), tf.float32)
    if labelled:
        signature = (signature, tf.TensorSpec((), tf.int32))
    dataset = tf.data.Dataset.from_generator(examples, output_signature=signature)
    return dataset.apply(tf.data.experimental.assert_cardinality(len(rows)))


def make_validation_dataset():
    return image_dataset(val_rows, labelled=True).batch(BATCH_SIZE).prefetch(1)


def make_training_dataset(epoch):
    # Every retained mango training identity is presented once per epoch.
    # A separately shuffled OE stream supplies one batch per mango batch and
    # cycles as needed; its count never changes class weights or mango accuracy.
    mango_order = list(train_rows)
    oe_order = list(approved_oe_rows)
    random.Random(SEED + epoch).shuffle(mango_order)
    random.Random(SEED + 1000 + epoch).shuffle(oe_order)
    mango_layers = augmentation_layers(SEED + 10000 + 10 * epoch)
    oe_layers = augmentation_layers(SEED + 20000 + 10 * epoch)
    mango = image_dataset(mango_order, labelled=True).batch(BATCH_SIZE)
    mango = mango.map(lambda x, y: (augment_batch(x, mango_layers), y), num_parallel_calls=1)
    oe = image_dataset(oe_order).batch(BATCH_SIZE).repeat()
    oe = oe.map(lambda x: augment_batch(x, oe_layers), num_parallel_calls=1)
    combined = tf.data.Dataset.zip((mango, oe))
    return combined.map(lambda labelled, outliers: ({'mango': labelled[0], 'oe': outliers}, labelled[1]), num_parallel_calls=1).prefetch(1)


fig, axes = plt.subplots(7, 8, figsize=(24, 23))
headings = ['Original', 'Flip', 'Rotation', 'Zoom', 'Shear', 'Brightness', 'Contrast 0.1', 'Combined']
for index, row in enumerate(example_rows):
    pixels = load_pixels(row['path'])[None]
    layers = augmentation_layers(SEED + 30000 + index * 10)
    views = [pixels] + [tf.clip_by_value(layer(pixels, training=True), 0, 255) for layer in layers]
    views.append(augment_batch(pixels, augmentation_layers(SEED + 40000 + index * 10)))
    for axis, heading, view in zip(axes[index], headings, views):
        axis.imshow(np.uint8(view.numpy()[0]))
        axis.set_title(heading)
        axis.set_xticks([])
        axis.set_yticks([])
    axes[index, 0].set_ylabel(row['label'].replace('_', ' '), fontsize=11)
fig.suptitle('V4.5.0 seven-class augmentation preview', fontsize=20)
fig.text(0.5, 0.01, 'Illustration-only validation originals. Training transforms preserve labels; contrast changes light/dark separation. No background replacement or localized lighting.', ha='center', fontsize=11)
fig.tight_layout(rect=(0, 0.025, 1, 0.97))
fig.savefig(FIGURE_DIR / 'augmentation_preview.png', dpi=150)
plt.show()
plt.close(fig)
print('Preview:', FIGURE_DIR / 'augmentation_preview.png')


In [ ]:
AUGMENTATION_REVIEW_COMPLETE = False
AUGMENTATION_REVIEW_NOTES = ''
if not AUGMENTATION_REVIEW_COMPLETE or not AUGMENTATION_REVIEW_NOTES.strip():
    raise RuntimeError('Inspect all six techniques and combined previews; approve only if important specimen/disease details remain visible.')
save_json_once(RUN_DIR / 'augmentation_review.json', {
    'complete': True, 'notes': AUGMENTATION_REVIEW_NOTES,
    'preview': 'figures/augmentation_preview.png',
    'previewSha256': file_sha256(FIGURE_DIR / 'augmentation_preview.png'),
    'trainingOnly': True,
})


## Step 9: Prepare a separate converter and test architecture export

**What:** Install a fresh CPU-only TensorFlow.js converter and convert a disposable architecture probe before training.

**Why:** Conversion failures should be discovered before the 15 training epochs. Converter packages must not replace the active Colab training kernel.

**How:** Create a new Python 3.11 environment using uv and explicit wheels-only pins. Run pip check and pip-audit. TensorFlow.js needs packaging 23.x, so the converter uses wheel 0.45.1. Its known wheel-unpack advisory and the setuptools 81 source-distribution advisory remain visible with exact unused-operation reviews; all other unresolved findings block. Convert a random-weight MobileNetV3-Large probe without training, then retain the logs.

**Expected result:** A fresh conversion preflight passes or stops with a named log. This checks conversion feasibility, not browser probability parity or model accuracy.

If this run stopped on the wheel 0.46.3 / packaging 23.2 installation conflict, replace only this Step 9 code cell with the corrected template cell and rerun it. It archives the two failed logs, records their hashes, and creates a fresh converter environment. Do not restart or rerun the image inventory. Completed preflight or frozen/trained runs cannot use this retry.


In [ ]:
CONVERTER_PACKAGES = [
    'pip==26.2.1', 'pip-audit==2.10.1', 'setuptools==81.0.0', 'wheel==0.45.1',
    'tensorflow==2.19.0', 'tf-keras==2.19.0',
    'tensorflow-decision-forests==1.12.0', 'tensorflow-hub==0.16.1',
    'ydf==0.11.0', 'protobuf==5.29.6', 'tensorflowjs==4.22.0',
    'keras==3.15.1', 'numpy==2.1.3', 'Pillow==12.3.0',
    'jax==0.4.35', 'jaxlib==0.4.35', 'flax==0.10.2', 'packaging==23.2',
    'idna==3.15', 'msgpack==1.2.1', 'requests==2.33.0',
    'urllib3==2.8.0', 'werkzeug==3.1.9',
]
# TensorFlow.js requires packaging 23.x. TFDF also requires wheel, whose
# compatible 0.45.1 release has a known unused unpack-operation advisory.
def prepare_converter_evidence(run_dir):
    """Preserve this specific failed install before retrying the converter."""
    run_dir = Path(run_dir)
    sealed_names = ('converter_preflight.json', 'plan.json', 'frozen_inputs.json',
                    'training_started.json', 'training_complete.json')
    if any((run_dir / name).exists() for name in sealed_names):
        raise RuntimeError('Converter preflight or training is already frozen; preserve the existing run.')

    evidence = run_dir / 'converter_preflight'
    archived = None
    if evidence.exists():
        expected_logs = {'environment-setup.log', 'package-installation.log'}
        existing = {path.name for path in evidence.iterdir()}
        if existing != expected_logs or not all((evidence / name).is_file() for name in expected_logs):
            raise RuntimeError('Existing converter evidence is not the reviewed installation failure; inspect it before retrying.')
        install_log = (evidence / 'package-installation.log').read_text(encoding='utf-8')
        conflict_details = ('No solution found', 'wheel==0.46.3', 'packaging>=24.0', 'packaging==23.2')
        if not all(detail in install_log for detail in conflict_details):
            raise RuntimeError('The existing converter failure has a different cause; preserve its logs and diagnose it first.')
        stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
        archived = evidence.with_name('converter_preflight.failed-' + stamp + '-' + uuid.uuid4().hex[:8])
        evidence.rename(archived)
        print('Failed converter logs preserved:', archived)

    evidence.mkdir(exist_ok=False)
    if archived is not None:
        preserved_logs = evidence / 'prior_failed_attempt'
        preserved_logs.mkdir()
        for path in sorted(archived.iterdir()):
            shutil.copy2(path, preserved_logs / path.name)
        save_json_once(evidence / 'retry-provenance.json', {
            'reason': 'Corrected incompatible wheel 0.46.3 and packaging 23.2 pins before training.',
            'previousAttempt': archived.relative_to(run_dir).as_posix(),
            'preservedLogs': {
                path.relative_to(run_dir).as_posix(): {'sha256': file_sha256(path), 'bytes': path.stat().st_size}
                for path in sorted(preserved_logs.iterdir())
            },
            'retryPackages': CONVERTER_PACKAGES,
        })
    return evidence


converter_evidence = prepare_converter_evidence(RUN_DIR)
# A new subprocess environment keeps the failed environment and active kernel intact.
converter_environment_root = Path('/content/mangoleaf-v450-converter') / (run_id + '-' + uuid.uuid4().hex[:8])
converter_environment_root.parent.mkdir(parents=True, exist_ok=True)
CONVERTER_ENVIRONMENT = dict(os.environ)
for key in ('PYTHONPATH', 'PYTHONHOME', 'VIRTUAL_ENV', 'TF_USE_LEGACY_KERAS'):
    CONVERTER_ENVIRONMENT.pop(key, None)
CONVERTER_ENVIRONMENT.update({'PYTHONNOUSERSITE': '1', 'CUDA_VISIBLE_DEVICES': '-1', 'TF_CPP_MIN_LOG_LEVEL': '2'})


def converter_run(command, log_name):
    result = subprocess.run(command, env=CONVERTER_ENVIRONMENT, capture_output=True, text=True, check=False)
    with (converter_evidence / log_name).open('x', encoding='utf-8') as handle:
        handle.write(result.stdout + '\n' + result.stderr)
    if result.returncode != 0:
        print((result.stdout + '\n' + result.stderr)[-4000:])
        raise RuntimeError('Converter preparation failed. Inspect: ' + str(converter_evidence / log_name))
    return result


converter_run([sys.executable, '-m', 'uv', 'venv', '--python', '3.11', str(converter_environment_root)], 'environment-setup.log')
CONVERTER_PYTHON = converter_environment_root / 'bin/python'
CONVERTER_COMMAND = converter_environment_root / 'bin/tensorflowjs_converter'
converter_run([
    sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(CONVERTER_PYTHON),
    '--only-binary=:all:', *CONVERTER_PACKAGES,
], 'package-installation.log')
converter_run([str(CONVERTER_PYTHON), '-m', 'pip', 'check'], 'pip-check.log')
freeze_result = converter_run([str(CONVERTER_PYTHON), '-m', 'pip', 'freeze', '--all'], 'freeze.log')
(converter_evidence / 'converter-runtime-lock.txt').write_text(freeze_result.stdout, encoding='utf-8')
audit_path = converter_evidence / 'audit.json'
audit = subprocess.run([
    str(CONVERTER_PYTHON), '-m', 'pip_audit', '--format=json',
    '--output', str(audit_path), '--progress-spinner=off',
], env=CONVERTER_ENVIRONMENT, capture_output=True, text=True, check=False)
(converter_evidence / 'audit.log').write_text(audit.stdout + '\n' + audit.stderr)
report = read_json(audit_path) if audit_path.is_file() else {}
dependencies = report.get('dependencies', [])
findings = [row for row in dependencies if row.get('vulns')]
skipped = [row for row in dependencies if row.get('skip_reason')]
known_sdist_aliases = {'GHSA-h35f-9h28-mq5c', 'CVE-2026-59890', 'PYSEC-2026-3447', 'BIT-setuptools-2026-59890'}
known_wheel_unpack_aliases = {'GHSA-8rrh-rw8j-w5fx', 'CVE-2026-24049'}
operation_reviews = []
blocking = []
for finding in findings:
    identities = [{vulnerability.get('id'), *vulnerability.get('aliases', [])} for vulnerability in finding['vulns']]
    reviewed_sdist = (
        canonicalize_name(finding['name']) == 'setuptools' and finding['version'] == '81.0.0'
        and identities and all(identity and identity <= known_sdist_aliases for identity in identities)
    )
    reviewed_wheel_unpack = (
        canonicalize_name(finding['name']) == 'wheel' and finding['version'] == '0.45.1'
        and identities and all(identity and identity <= known_wheel_unpack_aliases for identity in identities)
    )
    if reviewed_sdist:
        operation_reviews.append({
            'package': 'setuptools', 'version': '81.0.0', 'advisory': 'GHSA-h35f-9h28-mq5c',
            'status': 'unresolved advisory; affected source-distribution build/publish operation absent',
            'reason': 'TensorFlow Hub imports pkg_resources. Installation is wheels-only through uv; this environment only converts SavedModels and never builds or publishes source distributions.',
            'source': 'https://github.com/advisories/GHSA-h35f-9h28-mq5c',
        })
    elif reviewed_wheel_unpack:
        operation_reviews.append({
            'package': 'wheel', 'version': '0.45.1', 'advisory': 'GHSA-8rrh-rw8j-w5fx',
            'status': 'unresolved advisory; affected wheel unpack/RECORD permission operation absent',
            'reason': 'TFDF requires wheel and TensorFlow.js requires packaging 23.x. uv installs published binary wheels independently; this converter never calls wheel.cli.unpack or WheelFile permission extraction.',
            'source': 'https://github.com/advisories/GHSA-8rrh-rw8j-w5fx',
        })
    else:
        blocking.append(finding)
audit_allowed = bool(dependencies) and not skipped and not blocking and (
    audit.returncode == 0 and not findings or audit.returncode == 1 and bool(operation_reviews)
)
save_json_once(converter_evidence / 'audit-summary.json', {
    'findings': findings, 'skipped': skipped, 'returnCode': audit.returncode,
    'operationReviews': operation_reviews, 'executionAllowed': bool(audit_allowed),
    'status': 'passed' if not findings and audit_allowed else 'unresolved',
})
if not audit_allowed:
    raise RuntimeError('Converter audit contains unresolved execution findings or incomplete collection. Read ' + str(audit_path))
if operation_reviews:
    print('Known unused packaging-operation advisories remain visible:', audit_path)


def convert_saved_model(saved_model_path, export_path, log_path):
    result = subprocess.run([
        str(CONVERTER_COMMAND), '--input_format=tf_saved_model',
        '--output_format=tfjs_graph_model', '--signature_name=serving_default',
        '--saved_model_tags=serve', str(saved_model_path), str(export_path),
    ], env=CONVERTER_ENVIRONMENT, capture_output=True, text=True, check=False)
    Path(log_path).write_text(result.stdout + '\n' + result.stderr, encoding='utf-8')
    if result.returncode != 0:
        raise RuntimeError('Converter failed. Inspect ' + str(log_path))
    graph = read_json(export_path / 'model.json')
    if graph.get('format') != 'graph-model' or not graph.get('weightsManifest'):
        raise RuntimeError('Converter did not produce a TensorFlow.js graph model.')
    for group in graph['weightsManifest']:
        for relative in group['paths']:
            safe_child(export_path, relative).stat()
        if any(weight.get('quantization') is not None for weight in group['weights']):
            raise RuntimeError('This experiment requires unquantized float32 export.')


probe_backbone = tf.keras.applications.MobileNetV3Large(
    input_shape=(224, 224, 3), include_top=False, weights=None, include_preprocessing=True,
)
probe_inputs = tf.keras.Input((224, 224, 3), dtype=tf.float32, name='image')
probe_features = probe_backbone(probe_inputs, training=False)
probe_pooled = tf.keras.layers.GlobalAveragePooling2D()(probe_features)
probe_outputs = tf.keras.layers.Dense(7, activation='softmax')(probe_pooled)
probe_model = tf.keras.Model(probe_inputs, probe_outputs)
probe_model.export(converter_evidence / 'probe_saved_model', verbose=False)
convert_saved_model(converter_evidence / 'probe_saved_model', converter_evidence / 'probe_tfjs', converter_evidence / 'conversion.log')
save_json_once(RUN_DIR / 'converter_preflight.json', {
    'passed': True, 'trainingEpochs': 0, 'weights': 'random architecture probe',
    'tfjsFormat': 'graph-model', 'quantized': False,
    'browserProbabilityParity': 'pending actual browser comparison',
    'pins': CONVERTER_PACKAGES,
})
del probe_model, probe_backbone
tf.keras.utils.set_random_seed(SEED)
print('Fresh architecture conversion completed; no accuracy measured.')


## Step 10: Build fresh ImageNet features and calculate mango class weights

**What:** Create a plain seven-class MobileNetV3-Large classifier and calculate balanced training weights.

**Why:** ImageNet supplies general features, while a new head learns mango classes. Minority mango classes need explicit influence without relabeling, undersampling, or oversampling.

**How:** Use RGB 224×224 float32 values from 0 to 255; MobileNet performs its own rescaling. Freeze its backbone for stage 1. Compute each weight as training mango count divided by seven times that class count. OE photos never enter this formula.

**Expected result:** class_weights.json, initial_weights.json, and a plain Functional model. No earlier MangoLeaf trained weights are read.


In [ ]:
base_model = tf.keras.applications.MobileNetV3Large(
    input_shape=(224, 224, 3), include_top=False,
    weights='imagenet', include_preprocessing=True,
)
base_model.trainable = False
inputs = tf.keras.Input((224, 224, 3), dtype=tf.float32, name='image')
features = base_model(inputs, training=False)
pooled = tf.keras.layers.GlobalAveragePooling2D(name='global_average_pool')(features)
pooled = tf.keras.layers.Dropout(0.2, name='head_dropout')(pooled)
logits = tf.keras.layers.Dense(len(CLASS_NAMES), name='class_logits')(pooled)
outputs = tf.keras.layers.Activation('softmax', name='class_probabilities')(logits)
model = tf.keras.Model(inputs, outputs, name='mangoleaf_v4_5_0_seven_class')
logits_model = tf.keras.Model(inputs, logits, name='mangoleaf_v4_5_0_logits')

train_labels = np.array([CLASS_NAMES.index(row['label']) for row in train_rows])
weights = compute_class_weight('balanced', classes=np.arange(len(CLASS_NAMES)), y=train_labels)
class_weight = {index: float(value) for index, value in enumerate(weights)}
save_json_once(RUN_DIR / 'class_weights.json', {CLASS_NAMES[index]: value for index, value in class_weight.items()})
initial_digest = hashlib.sha256()
for array in model.get_weights():
    initial_digest.update(str(array.shape).encode())
    initial_digest.update(array.tobytes())
save_json_once(RUN_DIR / 'initial_weights.json', {
    'sha256': initial_digest.hexdigest(), 'source': 'ImageNet backbone plus newly initialized head', 'seed': SEED,
})
print('Training-only weights:', {CLASS_NAMES[index]: value for index, value in class_weight.items()})
model.summary()


### Small mathematical and serialization checks before fitting

These checks do not train the mango model. A seven-logit toy example must show that the uniform OE objective has a finite gradient that reduces confidence concentration. A saved/reloaded plain classifier must reproduce a synthetic image prediction without importing the custom OE trainer. Passing them proves the local mathematical wiring and model format; it does not measure mango accuracy or real-object rejection.


In [ ]:
def uniform_exposure_loss(logits):
    return -tf.reduce_mean(tf.nn.log_softmax(logits, axis=-1), axis=-1)


toy_logits = tf.Variable([[4.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]])
before_loss = float(tf.reduce_mean(uniform_exposure_loss(toy_logits)))
before_maximum = float(tf.reduce_max(tf.nn.softmax(toy_logits)))
with tf.GradientTape() as tape:
    toy_loss = tf.reduce_mean(uniform_exposure_loss(toy_logits))
toy_gradient = tape.gradient(toy_loss, toy_logits)
if toy_gradient is None or not np.isfinite(toy_gradient.numpy()).all() or not np.any(np.abs(toy_gradient.numpy()) > 0):
    raise RuntimeError('OE uniform-target loss has no valid nonzero gradient.')
toy_logits.assign_sub(0.1 * toy_gradient)
after_loss = float(tf.reduce_mean(uniform_exposure_loss(toy_logits)))
after_maximum = float(tf.reduce_max(tf.nn.softmax(toy_logits)))
if not after_loss < before_loss or not after_maximum < before_maximum:
    raise RuntimeError('OE toy update did not reduce concentration as expected.')
plain_probe_path = RUN_DIR / 'runtime_preflight/plain_classifier_probe.keras'
model.save(plain_probe_path)
plain_probe = tf.keras.models.load_model(plain_probe_path, compile=False, safe_mode=True)
synthetic_image = tf.zeros((1, 224, 224, 3), dtype=tf.float32)
np.testing.assert_allclose(model(synthetic_image, training=False), plain_probe(synthetic_image, training=False), rtol=1e-4, atol=1e-5)
if plain_probe.output_shape[-1] != 7:
    raise RuntimeError('The ordinary inference classifier must have seven outputs.')
save_json_once(RUN_DIR / 'runtime_preflight/oe_mathematical_probe.json', {
    'trainedMangoEpochs': 0, 'uniformTargetClassCount': 7,
    'beforeLoss': before_loss, 'afterLoss': after_loss,
    'beforeMaximumProbability': before_maximum, 'afterMaximumProbability': after_maximum,
    'finiteNonzeroGradient': True, 'plainClassifierRoundTripPassed': True,
    'accuracyOrOodRejectionMeasured': False,
})
del plain_probe, toy_logits
print('OE gradient and plain inference-model round-trip probes passed; no mango epochs trained.')


## Step 11: Freeze the reviewed inputs and experimental plan

**What:** Seal the chosen recipe, original identities, split membership, review notes, and runtime evidence before fitting.

**Why:** A reproducible result needs its actual configuration and unchanged inputs, not just a version label.

**How:** Record OE loss weight 0.5 as a proposed hyperparameter, mango-only metrics, fixed learning rates, contrast setting, leakage decisions, and all dataset hashes. Freeze the 500 held-out OOD identities before any training. Changed inputs require a new run.

**Expected result:** plan.json and frozen_inputs.json identify one auditable experimental attempt. Thresholds and final evaluation remain pending.


In [ ]:
plan = {
    'schemaVersion': 1, 'experimentVersion': EXPERIMENT_VERSION, 'modelVersion': MODEL_VERSION,
    'classNames': CLASS_NAMES, 'seed': SEED, 'imageSize': list(IMAGE_SIZE), 'batchSize': BATCH_SIZE,
    'initialWeights': 'ImageNet MobileNetV3Large; no prior MangoLeaf model',
    'splitPolicy': manifest['splitPolicy'], 'splitFractionsTarget': [0.70, 0.15, 0.15],
    'stage1': {'epochs': STAGE1_EPOCHS, 'learningRate': STAGE1_LEARNING_RATE, 'backboneFrozen': True},
    'stage2': {'epochs': STAGE2_EPOCHS, 'learningRate': STAGE2_LEARNING_RATE, 'batchNormalizationFrozen': True},
    'modelSelection': 'Lowest clean unweighted mango validation loss in each stage; final checkpoint from stage 2',
    'augmentation': {'flip': 'horizontal_and_vertical', 'rotationDegrees': 30, 'zoom': 0.2, 'shear': 0.2, 'brightness': 0.2, 'contrast': CONTRAST_FACTOR},
    'oe': {
        'lossWeight': OE_LOSS_WEIGHT, 'target': 'uniform over seven mango classes',
        'loss': 'negative mean log-softmax over classes; expectation on approved OE training photos',
        'objective': 'mean(class_weight * mango cross-entropy) + lossWeight * mean(OE uniform cross-entropy)',
        'weightRationale': '0.5 is a starting research hyperparameter, not inferred from photo count or a proven MangoLeaf optimum',
        'sampling': 'One independently shuffled OE batch per mango batch, cycling as needed; reshuffle every epoch',
        'approvedCount': len(approved_oe_rows), 'accuracyIncludesOE': False,
        'classWeightsIncludeOE': False, 'temperatureIncludesOE': False,
    },
    'classWeightRule': 'Balanced weights from mango training labels only',
    'phashDistance': PHASH_DISTANCE, 'preprocessing': PREPROCESSING,
    'temperatureRule': 'Validation mango negative log likelihood only; divide classifier logits once',
    'heldOutOodCount': len(held_out_ood_rows), 'expectedOodCount': EXPECTED_OOD_IMAGE_COUNT,
    'validationExamples': [row['id'] for row in example_rows],
    'testPredictionsRun': False, 'thresholdsFitted': False,
}
save_json_once(RUN_DIR / 'plan.json', plan)
frozen_paths = [
    'plan.json', 'input_manifest.json', 'dataset_split.zip', 'class_names.json',
    'mango_inventory.json', 'oe_raw_inventory.json', 'oe_inventory.json',
    'oe_training_manifest.json', 'oe_dataset_manifest.json', 'held_out_ood_raw_inventory.json',
    'held_out_ood_inventory.json', 'held_out_ood_group_review.json',
    'reviewed_images.json', 'duplicate_audit.json', 'duplicate_review.json',
    'oe_duplicate_audit.json', 'oe_review.json', 'cross_pool_overlap_audit.json',
    'cross_pool_overlap_review.json', 'split_review.json', 'example_selection.json',
    'augmentation_review.json', 'figures/augmentation_preview.png',
    'runtime.json', 'converter_preflight.json', 'class_weights.json', 'initial_weights.json',
]
frozen_paths += [row['relativePath'] for row in approved_oe_rows]
frozen_paths += ['dataset/' + row['relativePath'] for row in records]
frozen_paths += [path.relative_to(RUN_DIR).as_posix() for directory in ('runtime_preflight', 'converter_preflight')
                 for path in sorted((RUN_DIR / directory).rglob('*')) if path.is_file()]
frozen_paths = sorted(set(frozen_paths))
frozen = {
    'configurationHash': object_sha256(plan),
    'artifacts': {relative: {'sha256': file_sha256(safe_child(RUN_DIR, relative)), 'bytes': safe_child(RUN_DIR, relative).stat().st_size}
                  for relative in frozen_paths},
}
save_json_once(RUN_DIR / 'frozen_inputs.json', frozen)


def verify_receipt_artifacts(receipt):
    for relative, expected in receipt['artifacts'].items():
        artifact = safe_child(RUN_DIR, relative)
        if artifact.stat().st_size != expected['bytes'] or file_sha256(artifact) != expected['sha256']:
            raise RuntimeError('Sealed artifact changed: ' + relative)


print('Frozen configuration:', frozen['configurationHash'])
print('Frozen artifacts:', len(frozen['artifacts']))


## Step 12: Train mango classification and OE uncertainty for five plus ten epochs

**What:** Optimize two separate objectives while recording mango-only accuracy, supervised loss, OE loss, total loss, and actual learning rate.

**Why:** OE photos have no mango ground-truth label and must not inflate class counts or accuracy. Clean mango validation determines checkpoint selection.

**How:** For each batch, average class-weighted mango cross-entropy and add 0.5 times uniform-target OE cross-entropy. Train the head for five epochs at 0.001, restore its best checkpoint, then fine-tune for ten at 0.00001 with BatchNormalization frozen. Final selection is the best clean validation loss in stage 2. Export the plain Functional classifier, not the custom training wrapper.

**Expected result:** Fifteen real epoch records and two best checkpoints. No test/OOD reporting predictions occur; incomplete runs cannot be sealed as finished.


In [ ]:
def verify_initial_training_state(classifier, balanced_weights, weight_by_index):
    # Verify the live objects consumed by fit, not only the frozen JSON files.
    # A failed rerun of model creation must not silently replace this run's head.
    initial_digest = hashlib.sha256()
    for array in classifier.get_weights():
        initial_digest.update(str(array.shape).encode())
        initial_digest.update(array.tobytes())
    if initial_digest.hexdigest() != read_json(RUN_DIR / 'initial_weights.json')['sha256']:
        raise RuntimeError('Live initial model weights differ from the frozen identity; create a new run.')

    counts = np.array([sum(row['label'] == label for row in train_rows) for label in CLASS_NAMES], dtype=np.float64)
    if np.any(counts <= 0):
        raise ValueError('Every mango class needs at least one training photo.')
    expected = len(train_rows) / (len(CLASS_NAMES) * counts)
    live = np.asarray(balanced_weights, dtype=np.float64)
    sealed = read_json(RUN_DIR / 'class_weights.json')
    if set(sealed) != set(CLASS_NAMES) or set(weight_by_index) != set(range(len(CLASS_NAMES))):
        raise RuntimeError('Class-weight identities differ from the agreed seven classes.')
    sealed_vector = np.array([sealed[label] for label in CLASS_NAMES], dtype=np.float64)
    mapping_vector = np.array([weight_by_index[index] for index in range(len(CLASS_NAMES))], dtype=np.float64)
    for values in [live, sealed_vector, mapping_vector]:
        if values.shape != expected.shape or not np.isfinite(values).all() or not np.allclose(values, expected, rtol=1e-12, atol=0):
            raise RuntimeError('Live or sealed class weights differ from fresh balanced mango counts; create a new run.')


class OETrainer(tf.keras.Model):
    def __init__(self, classifier_logits, balanced_weights, oe_weight):
        super().__init__(name='mangoleaf_oe_training_wrapper')
        self.classifier_logits = classifier_logits
        self.balanced_weights = tf.constant(balanced_weights, dtype=tf.float32)
        self.oe_weight = tf.constant(oe_weight, dtype=tf.float32)
        self.total_tracker = tf.keras.metrics.Mean(name='loss')
        self.mango_tracker = tf.keras.metrics.Mean(name='mango_loss')
        self.oe_tracker = tf.keras.metrics.Mean(name='oe_loss')
        self.accuracy_tracker = tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')

    def call(self, inputs, training=False):
        # Keras uses call() for symbolic shape/build checks before fit. The
        # custom train_step still computes both mango and OE objectives.
        pixels = inputs['mango'] if isinstance(inputs, dict) else inputs
        return self.classifier_logits(pixels, training=training)

    @property
    def metrics(self):
        return [self.total_tracker, self.mango_tracker, self.oe_tracker, self.accuracy_tracker]

    def train_step(self, data):
        inputs, labels, _ = tf.keras.utils.unpack_x_y_sample_weight(data)
        with tf.GradientTape() as tape:
            mango_logits = self.classifier_logits(inputs['mango'], training=True)
            outlier_logits = self.classifier_logits(inputs['oe'], training=True)
            original_losses = tf.keras.losses.sparse_categorical_crossentropy(labels, mango_logits, from_logits=True)
            per_photo_weighted = original_losses * tf.gather(self.balanced_weights, labels)
            supervised_loss = tf.reduce_mean(per_photo_weighted)
            # A uniform target uses every output class equally. OE has no
            # fabricated Healthy Leaf label and no additional output class.
            per_outlier_loss = uniform_exposure_loss(outlier_logits)
            oe_loss = tf.reduce_mean(per_outlier_loss)
            total_loss = supervised_loss + self.oe_weight * oe_loss
        variables = self.classifier_logits.trainable_variables
        gradients = tape.gradient(total_loss, variables)
        pairs = [(gradient, variable) for gradient, variable in zip(gradients, variables) if gradient is not None]
        if len(pairs) != len(variables):
            raise RuntimeError('A trainable classifier variable has no gradient.')
        self.optimizer.apply_gradients(pairs)
        self.total_tracker.update_state(total_loss, sample_weight=tf.cast(tf.shape(labels)[0], tf.float32))
        self.mango_tracker.update_state(per_photo_weighted)
        self.oe_tracker.update_state(per_outlier_loss)
        self.accuracy_tracker.update_state(labels, tf.nn.softmax(mango_logits))
        return {metric.name: metric.result() for metric in self.metrics}

    def test_step(self, data):
        pixels, labels, _ = tf.keras.utils.unpack_x_y_sample_weight(data)
        logits = self.classifier_logits(pixels, training=False)
        # Validation loss is clean, unweighted mango cross-entropy. It never
        # includes OE uncertainty loss or balanced training class weights.
        clean_losses = tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)
        self.total_tracker.update_state(clean_losses)
        self.mango_tracker.update_state(clean_losses)
        self.oe_tracker.update_state(0.0)
        self.accuracy_tracker.update_state(labels, tf.nn.softmax(logits))
        return {metric.name: metric.result() for metric in self.metrics}


if (RUN_DIR / 'training_started.json').exists():
    raise RuntimeError('This run already started fitting. Preserve an interrupted attempt; create a new run to retrain.')
if object_sha256(plan) != frozen['configurationHash']:
    raise RuntimeError('Plan changed after freeze.')
live_recipe = {'flip': 'horizontal_and_vertical', 'rotationDegrees': 30, 'zoom': 0.2, 'shear': 0.2, 'brightness': 0.2, 'contrast': CONTRAST_FACTOR}
if plan['augmentation'] != live_recipe or plan['oe']['lossWeight'] != OE_LOSS_WEIGHT:
    raise RuntimeError('Augmentation or OE settings changed after freeze; create a new run.')
if plan['stage1'] != {'epochs': STAGE1_EPOCHS, 'learningRate': STAGE1_LEARNING_RATE, 'backboneFrozen': True} or plan['stage2'] != {'epochs': STAGE2_EPOCHS, 'learningRate': STAGE2_LEARNING_RATE, 'batchNormalizationFrozen': True}:
    raise RuntimeError('Stage settings changed after freeze; create a new run.')
if portable_rows(records) != manifest['images'] or read_json(RUN_DIR / 'input_manifest.json') != manifest:
    raise RuntimeError('Live mango identities/order differ from the frozen manifest.')
if portable_rows(train_rows) != [row for row in manifest['images'] if row['subset'] == 'train'] or portable_rows(val_rows) != [row for row in manifest['images'] if row['subset'] == 'val']:
    raise RuntimeError('Live training or validation rows differ from the frozen mango subsets.')
if portable_rows(approved_oe_rows) != read_json(RUN_DIR / 'oe_dataset_manifest.json')['images']:
    raise RuntimeError('Live OE training identities/order differ from the frozen manifest.')
verify_receipt_artifacts(frozen)
verify_image_rows(records)
verify_image_rows(approved_oe_rows)
verify_image_rows(held_out_ood_rows)
verify_initial_training_state(model, weights, class_weight)
if logits_model.get_layer('class_logits') is not model.get_layer('class_logits') or logits_model.get_layer(base_model.name) is not base_model:
    raise RuntimeError('The live training logits are not connected to the verified plain classifier.')
save_json_once(RUN_DIR / 'training_started.json', {
    'startedAt': datetime.now(timezone.utc).isoformat(), 'configurationHash': frozen['configurationHash'],
})

training_history = []
completed_epochs = 0
for stage, epoch_count, learning_rate in (
    (1, STAGE1_EPOCHS, STAGE1_LEARNING_RATE),
    (2, STAGE2_EPOCHS, STAGE2_LEARNING_RATE),
):
    if stage == 2:
        base_model.trainable = True
        for layer in base_model.layers:
            if isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False
    trainer = OETrainer(logits_model, weights, OE_LOSS_WEIGHT)
    trainer.compile(optimizer=tf.keras.optimizers.Adam(learning_rate), jit_compile=False)
    best_loss, best_epoch = math.inf, None
    checkpoint = RUN_DIR / f'stage{stage}_best.weights.h5'
    for stage_epoch in range(epoch_count):
        print(f'Stage {stage}; overall epoch {completed_epochs + 1}/{STAGE1_EPOCHS + STAGE2_EPOCHS}')
        history = trainer.fit(
            make_training_dataset(completed_epochs), validation_data=make_validation_dataset(),
            epochs=1, shuffle=False, verbose=2,
        )
        metrics = {name: float(values[-1]) for name, values in history.history.items()}
        metrics['learning_rate'] = float(tf.keras.backend.get_value(trainer.optimizer.learning_rate))
        if not all(math.isfinite(value) for value in metrics.values()):
            raise RuntimeError('Non-finite training metric; do not export this incomplete run.')
        completed_epochs += 1
        row = {
            'stage': stage, 'epoch': completed_epochs, **metrics,
            'mangoTrainingCount': len(train_rows),
            'oeBatches': math.ceil(len(train_rows) / BATCH_SIZE),
        }
        training_history.append(row)
        save_json_once(RUN_DIR / f'epoch_{completed_epochs:02d}.json', row)
        if metrics['val_loss'] < best_loss:
            best_loss, best_epoch = metrics['val_loss'], completed_epochs
            model.save_weights(checkpoint)
    model.load_weights(checkpoint)
    save_json_once(RUN_DIR / f'stage{stage}_selection.json', {
        'stage': stage, 'selectedEpoch': best_epoch, 'validationLoss': best_loss,
        'criterion': 'clean unweighted mango validation cross-entropy',
    })

selected_model = model
selected_model.save(RUN_DIR / 'model_selected.keras')
save_json_once(RUN_DIR / 'training_history.json', training_history)
verify_receipt_artifacts(frozen)
verify_image_rows(records)
verify_image_rows(approved_oe_rows)
fitted_paths = list(frozen['artifacts']) + [
    'frozen_inputs.json', 'training_started.json', 'model_selected.keras', 'training_history.json',
    'stage1_best.weights.h5', 'stage2_best.weights.h5', 'stage1_selection.json', 'stage2_selection.json',
]
save_json_once(RUN_DIR / 'training_state.json', {
    'status': 'fitted; calibration/export pending', 'configurationHash': frozen['configurationHash'],
    'completedEpochs': completed_epochs, 'testPredictionsRun': False,
    'artifacts': {relative: {'sha256': file_sha256(safe_child(RUN_DIR, relative)), 'bytes': safe_child(RUN_DIR, relative).stat().st_size}
                  for relative in fitted_paths},
})
print('Selected stage 2 checkpoint:', read_json(RUN_DIR / 'stage2_selection.json'))


## Step 13: Fit temperature on clean mango validation photos only

**What:** Fit one confidence-calibration value without OE, OOD, or mango test photos.

**Why:** OE changes output scores, so old temperatures cannot be reused. Calibration does not establish unsupported-input rejection.

**How:** Predict validation logits in frozen manifest order, minimize unweighted negative log likelihood, keep temperature 1 if no improvement, then divide the final Dense weights and biases once. Verify class predictions remain unchanged and reload the saved plain model.

**Expected result:** model_calibrated.keras, temperature.json, validation_logits.npy, and validation_probabilities.npy. Temperature is already folded into weights.


In [ ]:
verify_receipt_artifacts(read_json(RUN_DIR / 'training_state.json'))
if (RUN_DIR / 'calibration_complete.json').exists():
    raise RuntimeError('Calibration is already sealed. Read its artifacts rather than refitting or folding temperature twice.')


def predict_records(classifier, rows):
    return classifier.predict(image_dataset(rows).batch(BATCH_SIZE).prefetch(1), verbose=0)


logits_predictor = tf.keras.Model(selected_model.inputs, selected_model.get_layer('class_logits').output)
validation_logits = predict_records(logits_predictor, val_rows).astype(np.float64)
validation_labels = np.array([CLASS_NAMES.index(row['label']) for row in val_rows])


def validation_nll(temperature):
    scaled = validation_logits / temperature
    shifted = scaled - scaled.max(axis=1, keepdims=True)
    log_probabilities = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
    return float(-np.mean(log_probabilities[np.arange(len(validation_labels)), validation_labels]))


calibration = minimize_scalar(validation_nll, bounds=(0.25, 20.0), method='bounded')
if not calibration.success or not math.isfinite(calibration.fun):
    raise RuntimeError('Validation-only temperature fit failed.')
temperature = float(calibration.x) if calibration.fun < validation_nll(1.0) else 1.0
calibrated_model = tf.keras.models.clone_model(selected_model)
calibrated_model.set_weights(selected_model.get_weights())
head = calibrated_model.get_layer('class_logits')
head_weights, head_bias = head.get_weights()
head.set_weights([head_weights / temperature, head_bias / temperature])
probabilities = predict_records(calibrated_model, val_rows)
original_probabilities = predict_records(selected_model, val_rows)
if not np.array_equal(original_probabilities.argmax(axis=1), probabilities.argmax(axis=1)):
    raise RuntimeError('Temperature folding changed a validation class prediction.')
if not np.isfinite(probabilities).all() or not np.allclose(probabilities.sum(axis=1), 1, atol=1e-5):
    raise RuntimeError('Invalid calibrated validation probabilities.')
calibrated_model.save(RUN_DIR / 'model_calibrated.keras')
np.save(RUN_DIR / 'validation_logits.npy', validation_logits)
np.save(RUN_DIR / 'validation_probabilities.npy', probabilities)
reloaded = tf.keras.models.load_model(RUN_DIR / 'model_calibrated.keras', compile=False, safe_mode=True)
np.testing.assert_allclose(probabilities, predict_records(reloaded, val_rows), rtol=1e-4, atol=1e-5)
temperature_record = {
    'temperature': temperature, 'temperatureFoldedIntoWeights': True, 'foldCount': 1,
    'fitSubset': 'val', 'fitIncludesOE': False, 'fitIncludesOOD': False,
    'validationCount': len(val_rows), 'beforeNll': validation_nll(1.0), 'afterNll': validation_nll(temperature),
    'searchBounds': [0.25, 20.0], 'keepUnitTemperatureUnlessImproved': True,
    'validationProbabilityOrder': 'input_manifest.json images subset=val in listed order',
}
save_json_once(RUN_DIR / 'temperature.json', temperature_record)
calibration_paths = ['model_calibrated.keras', 'validation_logits.npy', 'validation_probabilities.npy', 'temperature.json']
save_json_once(RUN_DIR / 'calibration_complete.json', {
    'configurationHash': frozen['configurationHash'],
    'artifacts': {relative: {'sha256': file_sha256(RUN_DIR / relative), 'bytes': (RUN_DIR / relative).stat().st_size}
                  for relative in calibration_paths},
})
print('Validation-fitted temperature:', temperature)


## Step 14: Export a plain calibrated classifier without quantization

**What:** Convert the saved seven-softmax classifier to a TensorFlow.js graph and float32 shards.

**Why:** The browser needs ordinary inference assets, not the OE training wrapper or augmentation code. Temperature must not be applied a second time.

**How:** Export this run's calibrated Functional model to SavedModel and convert it in the reviewed CPU environment. Preserve a unique failed attempt instead of overwriting output. Metadata records preprocessing and class order but deliberately omits rejection thresholds.

**Expected result:** tfjs_model/model.json, model_config.json, shards, and export_complete.json. Actual browser probability parity remains pending.


In [ ]:
verify_receipt_artifacts(read_json(RUN_DIR / 'calibration_complete.json'))
destination = RUN_DIR / 'tfjs_model'
if destination.exists() or (RUN_DIR / 'export_complete.json').exists():
    raise RuntimeError('Export output already exists. Preserve it; never silently overwrite a sealed or partial attempt.')
attempt = RUN_DIR / 'export_attempts' / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])
attempt.mkdir(parents=True, exist_ok=False)
saved_model_path = attempt / 'saved_model'
exported_path = attempt / 'tfjs_model'
calibrated_model.export(saved_model_path, verbose=False)
convert_saved_model(saved_model_path, exported_path, attempt / 'conversion.log')
configuration = {
    'modelVersion': MODEL_VERSION, 'experimentVersion': EXPERIMENT_VERSION,
    'status': 'experimental candidate; rejection thresholds not fitted',
    'classNames': CLASS_NAMES, 'classes': CLASS_NAMES,
    'inputSize': [224, 224], 'inputRange': [0, 255], 'inputDtype': 'float32',
    'preprocessing': PREPROCESSING, 'temperature': temperature_record['temperature'],
    'temperatureFoldedIntoWeights': True, 'quantized': False,
    'browserParityStatus': 'pending actual browser comparison before adoption',
    'trainingObjective': 'mango supervision plus uniform-target Outlier Exposure',
}
save_json_once(exported_path / 'model_config.json', configuration)
shutil.copytree(exported_path, destination)
save_json_once(RUN_DIR / 'export_complete.json', {
    'format': 'graph-model', 'quantized': False,
    'attempt': attempt.relative_to(RUN_DIR).as_posix(),
    'calibratedModelSha256': file_sha256(RUN_DIR / 'model_calibrated.keras'),
    'artifacts': {path.relative_to(RUN_DIR).as_posix(): {'sha256': file_sha256(path), 'bytes': path.stat().st_size}
                  for path in sorted(destination.rglob('*')) if path.is_file()},
})
print('Candidate export:', destination)


## Step 15: Inspect connected Grad-CAM maps on the same seven originals

**What:** Draw a true-class heatmap and a calibrated prediction caption for each fixed preview identity.

**Why:** A real gradient path through the selected model is needed; a colorful overlay alone is not evidence of causal disease recognition or accuracy.

**How:** Compute backbone features with inference behavior, watch those tensors, and execute the actual trained head inside GradientTape. Display original, map, and overlay; leave wrong predictions and absent positive signal visible. The source identities match augmentation_preview.png exactly.

**Expected result:** grad_cam.png and figure_manifest.json carry actual v4.5.0 predictions, source hashes, layer names, labels, and interpretation limits.


In [ ]:
def connected_grad_cam(classifier, batch, target_index):
    backbone = next((layer for layer in classifier.layers if isinstance(layer, tf.keras.Model)), None)
    if backbone is None:
        raise ValueError('A connected MobileNet backbone is required for Grad-CAM.')
    head_layers = classifier.layers[classifier.layers.index(backbone) + 1:]
    dense = next((layer for layer in head_layers if isinstance(layer, tf.keras.layers.Dense)), None)
    if dense is None or dense.get_config()['activation'] != 'linear':
        raise ValueError('Grad-CAM requires the real linear classifier logits.')
    features = backbone(batch, training=False)
    if features.shape.rank != 4:
        raise ValueError('Backbone output must be a spatial feature map.')
    with tf.GradientTape() as tape:
        tape.watch(features)
        output = features
        for layer in head_layers:
            output = layer(output, training=False)
            if layer is dense:
                logits = output
                break
        score = logits[:, target_index]
    gradients = tape.gradient(score, features)
    if gradients is None or not np.isfinite(gradients.numpy()).all():
        raise RuntimeError('Grad-CAM gradients are missing or non-finite.')
    weights = tf.reduce_mean(gradients, axis=(1, 2))
    heatmap = tf.maximum(tf.reduce_sum(features[0] * weights[0], axis=-1), 0)
    maximum = float(tf.reduce_max(heatmap))
    if maximum > 0:
        heatmap = heatmap / maximum
    resized = tf.image.resize(heatmap[..., None], batch.shape[1:3], method='bilinear')[..., 0]
    return resized.numpy(), maximum > 0, backbone.name, dense.name


figure_manifest = {
    'experimentVersion': EXPERIMENT_VERSION, 'subset': 'val',
    'illustrationsEnterTrainingOrScoring': False,
    'selectionBeforePredictions': True, 'examples': [],
}
fig, axes = plt.subplots(7, 3, figsize=(13, 25))
for index, row in enumerate(example_rows):
    pixels = load_pixels(row['path']).numpy()
    batch = tf.convert_to_tensor(pixels[None], dtype=tf.float32)
    probabilities = calibrated_model(batch, training=False).numpy()[0]
    predicted_index = int(np.argmax(probabilities))
    target_index = CLASS_NAMES.index(row['label'])
    heatmap, has_signal, backbone_name, logits_name = connected_grad_cam(selected_model, batch, target_index)
    colored = plt.colormaps['jet'](heatmap)[..., :3]
    overlay = np.clip(0.65 * pixels / 255 + 0.35 * colored, 0, 1) if has_signal else pixels / 255
    headings = [
        'True: ' + row['label'].replace('_', ' '),
        'True-class Grad-CAM' if has_signal else 'No positive Grad-CAM signal',
        'Predicted: ' + CLASS_NAMES[predicted_index].replace('_', ' ') + f"\nCalibrated confidence: {probabilities[predicted_index]:.1%}",
    ]
    for axis, view, heading in zip(axes[index], [pixels / 255, heatmap, overlay], headings):
        axis.imshow(view, cmap='jet' if view.ndim == 2 else None, vmin=0, vmax=1)
        axis.set_title(heading, fontsize=11)
        axis.axis('off')
    figure_manifest['examples'].append({
        'id': row['id'], 'label': row['label'], 'sourceRelativePath': row['sourceRelativePath'],
        'sha256': row['sha256'], 'groupId': row['groupId'], 'subset': row['subset'],
        'gradCamTargetClass': row['label'], 'predictedClass': CLASS_NAMES[predicted_index],
        'calibratedConfidence': float(probabilities[predicted_index]),
        'backboneLayer': backbone_name, 'logitsLayer': logits_name, 'hasPositiveHeatmap': has_signal,
    })
fig.suptitle('V4.5.0 seven-class validation illustration Grad-CAM', fontsize=18)
fig.text(0.5, 0.01, 'Warmer colors show stronger positive contribution to the true-class logit. Heatmaps do not prove correct diagnosis; predictions and confidence remain visible.', ha='center', fontsize=10)
fig.tight_layout(rect=(0, 0.025, 1, 0.98))
fig.savefig(FIGURE_DIR / 'grad_cam.png', dpi=150)
plt.show()
plt.close(fig)


## Step 16: Plot actual training and validation curves side by side

**What:** Create one figure with accuracy on the left and mango loss on the right across the actual fifteen epochs.

**Why:** The requested historical layout is useful, but its old numbers must never be copied into a new experiment. Training and validation loss have different definitions here.

**How:** Read saved epoch values without smoothing. Show the stage boundary between epochs 5 and 6 and label learning rates 0.001 and 0.00001. The loss panel displays weighted augmented mango training loss and clean unweighted validation loss; OE/total objectives remain separately logged.

**Expected result:** training_curves.png has a v4.5.0 title, true recorded values, stage/learning-rate labels, and a caption explaining loss comparability.


In [ ]:
history = read_json(RUN_DIR / 'training_history.json')
if [row['epoch'] for row in history] != list(range(1, STAGE1_EPOCHS + STAGE2_EPOCHS + 1)):
    raise RuntimeError('Training history must contain all fifteen epochs in order.')
epochs = [row['epoch'] for row in history]
fig, axes = plt.subplots(1, 2, figsize=(19, 6.5))
axes[0].plot(epochs, [row['accuracy'] for row in history], 'o-', label='Training mango')
axes[0].plot(epochs, [row['val_accuracy'] for row in history], 's-', label='Validation mango')
axes[0].set_title('Accuracy: stage 1 (frozen base), then stage 2 (fine-tuning)')
axes[0].set_ylabel('Accuracy')
axes[1].plot(epochs, [row['mango_loss'] for row in history], 'o-', label='Training mango (weighted)')
axes[1].plot(epochs, [row['val_loss'] for row in history], 's-', label='Validation mango (unweighted)')
axes[1].set_title('Mango cross-entropy: weighted training and clean validation')
axes[1].set_ylabel('Loss')
for axis in axes:
    axis.axvline(STAGE1_EPOCHS + 0.5, color='gray', linestyle='--', label='Stage 2 begins')
    axis.set_xlabel('Epoch')
    axis.grid(alpha=0.25)
    axis.legend()
    axis.text(0.02, 0.03, 'Stage 1: LR 0.001 | Stage 2: LR 0.00001', transform=axis.transAxes, fontsize=10)
fig.suptitle('V4.5.0 training history: 5 epochs at 1e-3, then 10 epochs at 1e-5', fontsize=18)
fig.text(0.5, 0.012, 'Accuracy excludes OE. Training mango loss includes class weights and augmentation; validation is clean and unweighted. OE and total loss are recorded separately in training_history.json.', ha='center', fontsize=10)
fig.tight_layout(rect=(0, 0.055, 1, 0.94))
fig.savefig(FIGURE_DIR / 'training_curves.png', dpi=160)
plt.show()
plt.close(fig)
figure_manifest['figures'] = {
    name: {'sha256': file_sha256(FIGURE_DIR / name), 'bytes': (FIGURE_DIR / name).stat().st_size}
    for name in ('augmentation_preview.png', 'grad_cam.png', 'training_curves.png')
}
figure_manifest['lossPanel'] = 'Weighted augmented mango training loss versus clean unweighted mango validation loss; not total OE objective'
save_json_once(RUN_DIR / 'figure_manifest.json', figure_manifest)


## Step 17: Seal the completed training handoff for threshold fitting

**What:** Write a completion receipt covering this run's exact model, calibration, split, reviews, and figures.

**Why:** The threshold notebook must verify one completed v4.5.0 attempt before fitting gates or scoring any untouched test subset.

**How:** Recheck fitted/calibration/export hashes, complete epoch count, every copied image, and all 500 original held-out OOD identities. Include only real files in the receipt; Google Sheets placeholders cannot be hashed as ordinary CSV evidence. Save the exact run path.

**Expected result:** training_complete.json declares training/export complete but rejection fitting, final test evaluation, browser parity, and adoption still pending.


In [ ]:
if (RUN_DIR / 'training_complete.json').exists():
    raise RuntimeError('This run is already sealed; read the receipt without overwriting it.')
for receipt_name in ('training_state.json', 'calibration_complete.json', 'export_complete.json'):
    verify_receipt_artifacts(read_json(RUN_DIR / receipt_name))
fitted = read_json(RUN_DIR / 'training_state.json')
if fitted['completedEpochs'] != STAGE1_EPOCHS + STAGE2_EPOCHS:
    raise RuntimeError('All fifteen epochs must finish before sealing.')
verify_image_rows(records)
verify_image_rows(approved_oe_rows)
verify_image_rows(held_out_ood_rows)
current_ood = image_inventory(OOD_INPUT, 'ood')
original_ood_identities = {(row['sourceRelativePath'], row['sha256']) for row in held_out_ood_rows}
if {(row['sourceRelativePath'], row['sha256']) for row in current_ood} != original_ood_identities:
    raise RuntimeError('Held-out OOD membership changed after training inventory; preserve this run and create a new experiment.')
required = [
    'frozen_inputs.json', 'training_started.json', 'training_state.json',
    'model_selected.keras', 'model_calibrated.keras', 'temperature.json',
    'validation_logits.npy', 'validation_probabilities.npy', 'training_history.json',
    'calibration_complete.json', 'export_complete.json',
    'stage1_best.weights.h5', 'stage2_best.weights.h5', 'stage1_selection.json', 'stage2_selection.json',
    'figure_manifest.json', 'figures/augmentation_preview.png', 'figures/grad_cam.png', 'figures/training_curves.png',
]
required += list(frozen['artifacts'])
required += [f'epoch_{epoch:02d}.json' for epoch in range(1, STAGE1_EPOCHS + STAGE2_EPOCHS + 1)]
required += [path.relative_to(RUN_DIR).as_posix() for path in sorted((RUN_DIR / 'tfjs_model').rglob('*')) if path.is_file()]
required = sorted(set(required))
if any(PurePosixPath(relative).suffix == '.gsheet' for relative in required):
    raise ValueError('Google Sheets shortcuts are not source CSV files; use actual reviewed JSON/CSV evidence.')
receipt = {
    'schemaVersion': 1, 'experimentVersion': EXPERIMENT_VERSION, 'modelVersion': MODEL_VERSION,
    'status': 'training/export complete; experimental candidate; threshold fitting and final/browser evaluation pending',
    'completedAt': datetime.now(timezone.utc).isoformat(), 'classNames': CLASS_NAMES,
    'configurationHash': fitted['configurationHash'],
    'inputManifestSha256': file_sha256(RUN_DIR / 'input_manifest.json'),
    'calibratedTemperature': temperature_record['temperature'],
    'temperatureFoldedIntoWeights': True, 'foldedTemperature': True,
    'testPredictionsRun': False, 'browserParityPassed': False,
    'completedEpochs': fitted['completedEpochs'], 'selectedCheckpoint': read_json(RUN_DIR / 'stage2_selection.json'),
    'heldOutOodCount': len(held_out_ood_rows), 'oeTrainingCount': len(approved_oe_rows),
    'artifacts': {relative: {'sha256': file_sha256(safe_child(RUN_DIR, relative)), 'bytes': safe_child(RUN_DIR, relative).stat().st_size}
                  for relative in required},
}
save_json_once(RUN_DIR / 'training_complete.json', receipt)
print('Finished standalone v4.5.0 training run:', RUN_DIR)
print('Next: the fresh v4.5.0 threshold-fitting notebook using this exact run and the same 500 held-out OOD identities.')


## Sources and interpretation limits

- [Outlier Exposure paper](https://arxiv.org/html/1812.04606v3): auxiliary unsupported training examples and uniform-output encouragement. The chosen loss coefficient and data collection remain experimental for MangoLeaf.
- [Keras RandomContrast](https://keras.io/api/layers/preprocessing_layers/image_augmentation/random_contrast/): contrast changes deviations around channel means; factor 0.1 samples multipliers from 0.9 to 1.1.
- [TensorFlow MobileNetV3Large](https://www.tensorflow.org/api_docs/python/tf/keras/applications/MobileNetV3Large): embedded preprocessing expects float input in the 0–255 range.
- [TensorFlow transfer learning guide](https://www.tensorflow.org/tutorials/images/transfer_learning): freeze BatchNormalization behavior during fine-tuning and use a smaller learning rate.
- [TensorFlow.js conversion](https://www.tensorflow.org/js/guide/conversion): graph conversion and browser inference require compatible assets.
- [Known converter setuptools advisory](https://github.com/advisories/GHSA-h35f-9h28-mq5c): retained explicitly for the unused source-distribution build/publish operation; new or broader findings block execution.

The original 200 OOD photos have appeared in earlier experiments. Adding 300 fresh images does not erase that reuse: report historical and new-image results separately in threshold fitting. No current template claims independent performance from those reused photos, automatic production adoption, or verified phone behavior. Grad-CAM and synthetic probes explain wiring; they do not prove field accuracy.
